图4，两个代码，先运行第一个后，第二个能画图

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys
from matplotlib.path import Path
from matplotlib.lines import Line2D

# ===================== 0. 样式配置 (Arial 18pt) =====================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 18,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold"
})

# ===================== 1. 投影与范围 =====================
map_proj  = ccrs.LambertConformal(central_longitude=-41.5, central_latitude=71)
data_proj = ccrs.PlateCarree()

# ★关键：放大extent，保证格陵兰NW/NE完整
extent = [-75, -10, 58.0, 85.0]  # [lonW, lonE, latS, latN]

def clip_to_extent(ax, extent, crs, n=260):
    lon0, lon1, lat0, lat1 = extent
    lons = np.linspace(lon0, lon1, n)
    lats = np.linspace(lat0, lat1, n)

    bottom = np.column_stack([lons, np.full_like(lons, lat0)])
    right  = np.column_stack([np.full_like(lats, lon1), lats])
    top    = np.column_stack([lons[::-1], np.full_like(lons, lat1)])
    left   = np.column_stack([np.full_like(lats, lon0), lats[::-1]])

    ring = np.vstack([bottom, right, top, left])
    ax.set_boundary(Path(ring), transform=crs)

# ===================== 2. 数据加载与 JJA 气候态均值 =====================
def load_and_preprocess_means_850(path, extent):
    try:
        ds = xr.open_dataset(path, chunks={'valid_time': 100})

        # 统一坐标名
        if 'lat' in ds.coords:
            ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

        # 经度若是0–360，转为-180–180（避免slice(-75,-10)切不到）
        if ds.longitude.max() > 180:
            ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
            ds = ds.sortby("longitude")

        # 自动识别纬度降序
        is_descending = bool(ds.latitude[0] > ds.latitude[-1])
        lonW, lonE, latS, latN = extent
        lat_slice = slice(latN, latS) if is_descending else slice(latS, latN)

        ds_sub = ds[['z', 'u', 'v']].sel(
            pressure_level=850,
            longitude=slice(lonW, lonE),
            latitude=lat_slice
        )

        # z: geopotential -> height (m)
        ds_sub["z"] = ds_sub["z"] / 9.80665

        def jja_mean(start, end):
            sub = ds_sub.sel(valid_time=slice(start, end))
            sub = sub.sel(valid_time=sub.valid_time.dt.month.isin([6, 7, 8]))
            return sub.mean("valid_time").compute()

        print("正在计算 850 hPa 两段 JJA 均值：1950–1990, 2002–2024 ...")
        m_1950_1990 = jja_mean("1950-01-01", "1990-12-31")
        m_2002_2024 = jja_mean("2002-01-01", "2024-12-31")

        return m_1950_1990, m_2002_2024

    except Exception as e:
        print(f"数据初始化失败: {e}")
        sys.exit(1)


def load_10m_wind(path, extent):
    """加载10米风场数据"""
    try:
        ds = xr.open_dataset(path, chunks={'time': 100})

        # 统一坐标名
        if 'lat' in ds.coords:
            ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})
        if 'time' in ds.coords and 'valid_time' not in ds.coords:
            ds = ds.rename({'time': 'valid_time'})

        # 经度若是0–360，转为-180–180
        if ds.longitude.max() > 180:
            ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
            ds = ds.sortby("longitude")

        # 自动识别纬度降序
        is_descending = bool(ds.latitude[0] > ds.latitude[-1])
        lonW, lonE, latS, latN = extent
        lat_slice = slice(latN, latS) if is_descending else slice(latS, latN)

        ds_sub = ds[['u10', 'v10']].sel(
            longitude=slice(lonW, lonE),
            latitude=lat_slice
        )

        def jja_mean(start, end):
            sub = ds_sub.sel(valid_time=slice(start, end))
            sub = sub.sel(valid_time=sub.valid_time.dt.month.isin([6, 7, 8]))
            return sub.mean("valid_time").compute()

        print("正在计算 10m 风场两段 JJA 均值：1950–1990, 2002–2024 ...")
        w_1950_1990 = jja_mean("1950-01-01", "1990-12-31")
        w_2002_2024 = jja_mean("2002-01-01", "2024-12-31")

        return w_1950_1990, w_2002_2024

    except Exception as e:
        print(f"10m风场数据加载失败: {e}")
        sys.exit(1)

# ===================== 3. 读数据 =====================
PATH_850 = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\200500850UVZ.nc"
PATH_10M = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\ERA5_1950_2024_4_9_ForcedTime.nc"

m1, m2 = load_and_preprocess_means_850(PATH_850, extent)
w1, w2 = load_10m_wind(PATH_10M, extent)

# ===================== 4. 统一色标范围（跨两图一致） =====================
z_all = np.concatenate([
    m1['z'].values.ravel(),
    m2['z'].values.ravel()
])
z_all = z_all[np.isfinite(z_all)]
vmin, vmax = float(z_all.min()), float(z_all.max())

# ===================== 5. 绘图：一行两列（1450m 和 1430m 的年代际对比） =====================
fig = plt.figure(figsize=(16.0, 7))
gs = fig.add_gridspec(
    1, 2,
    wspace=0.10,
    left=0.04, right=0.98,
    top=0.90, bottom=0.18
)

# 定义两个年代的数据和样式
periods = [
    ("1950–1990", m1, 'blue', '-', 3.0),
    ("2002–2024", m2, 'red', '--', 3.0),
]

mappables = []
last_q = None

# 左图：1450m 等高线对比
ax1 = fig.add_subplot(gs[0, 0], projection=map_proj)
ax1.set_extent(extent, crs=data_proj)
clip_to_extent(ax1, extent, data_proj, n=260)

# 背景色标（使用第一个年代的数据）
im1 = ax1.pcolormesh(
    m1.longitude, m1.latitude, m1['z'],
    transform=data_proj, cmap='RdBu_r',
    vmin=vmin, vmax=vmax,
    shading='auto', rasterized=True, alpha=0.3
)
mappables.append(im1)

# 叠加两个年代的1450m等高线
for label, ds850, color, linestyle, lw in periods:
    cs = ax1.contour(
        ds850.longitude, ds850.latitude, ds850['z'],
        levels=[1450],
        colors=color, linewidths=lw, linestyles=linestyle,
        transform=data_proj, zorder=4
    )
    # 在等高线上标注
    ax1.clabel(cs, inline=True, fontsize=11, fmt='1450m')

# 风矢量（使用最近年代的10米风场数据）
q_ds = w2.isel(latitude=slice(None, None, 15), longitude=slice(None, None, 18))
q1 = ax1.quiver(
    q_ds.longitude.values, q_ds.latitude.values,
    q_ds['u10'].values, q_ds['v10'].values,
    transform=data_proj, color='grey',
    scale=55, width=0.007, headwidth=3, zorder=5, alpha=0.5
)

ax1.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
ax1.set_title("1450 m Geopotential Height - Decadal Changes", pad=10, weight='bold', fontsize=19)

# 右图：1430m 等高线对比
ax2 = fig.add_subplot(gs[0, 1], projection=map_proj)
ax2.set_extent(extent, crs=data_proj)
clip_to_extent(ax2, extent, data_proj, n=260)

# 背景色标（使用第一个年代的数据）
im2 = ax2.pcolormesh(
    m1.longitude, m1.latitude, m1['z'],
    transform=data_proj, cmap='RdBu_r',
    vmin=vmin, vmax=vmax,
    shading='auto', rasterized=True, alpha=0.3
)

# 叠加两个年代的1430m等高线
for label, ds850, color, linestyle, lw in periods:
    cs = ax2.contour(
        ds850.longitude, ds850.latitude, ds850['z'],
        levels=[1430],
        colors=color, linewidths=lw, linestyles=linestyle,
        transform=data_proj, zorder=4
    )
    # 在等高线上标注
    ax2.clabel(cs, inline=True, fontsize=11, fmt='1430m')

# 风矢量（使用最近年代的10米风场数据）
last_q = ax2.quiver(
    q_ds.longitude.values, q_ds.latitude.values,
    q_ds['u10'].values, q_ds['v10'].values,
    transform=data_proj, color='grey',
    scale=55, width=0.007, headwidth=3, zorder=5, alpha=0.5
)

ax2.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
ax2.set_title("1430 m Geopotential Height - Decadal Changes", pad=10, weight='bold', fontsize=19)

# ===================== 6. 添加图例 =====================
# 创建自定义图例
legend_elements = [
    Line2D([0], [0], color='blue', linewidth=3.0, linestyle='-', label='1950–1990'),
    Line2D([0], [0], color='red', linewidth=3.0, linestyle='--', label='2002–2024'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, 
           frameon=True, fontsize=16, bbox_to_anchor=(0.5, 0.08),
           edgecolor='black', fancybox=False)

# ===================== 7. 底部：色标 + 风矢量图例 =====================
cax = fig.add_axes([0.25, 0.01, 0.50, 0.04])
cb = plt.colorbar(mappables[0], cax=cax, orientation='horizontal')
cb.outline.set_linewidth(1)

# 只显示 min / max 两个刻度
cb.set_ticks([vmin, vmax])
cb.ax.set_xticklabels([f"{vmin:.0f}", f"{vmax:.0f}"])

cb.set_label("850 hPa | Geopotential Height (m)", fontsize=15, weight='normal')

if last_q is not None:
    plt.quiverkey(
        last_q,
        X=0.08, Y=0.01, U=5, label="5 m/s",
        labelpos='E', coordinates='figure',
        fontproperties={'family': 'Arial', 'size': 14, 'weight': 'normal'}
    )

plt.show()

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.path import Path
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from scipy.interpolate import interp1d
import math

# =====================================================================
# 0. Global rcParams
# =====================================================================
plt.rcParams.update({
    "font.family":        "Arial",
    "font.size":          8,
    "font.weight":        "bold",
    "axes.labelsize":     8,
    "axes.labelweight":   "bold",
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "legend.fontsize":    7,
    "axes.linewidth":     0.6,
    "xtick.direction":    "in",
    "ytick.direction":    "in",
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,
})

PANEL_FONTSIZE  = 12

ABC_TITLE_FS    = 10
ABC_CB_LABEL_FS = 8
ABC_CB_TICK_FS  = 8
ABC_LEG_FS      = 9

SEC_TITLE_FS    = 10
SEC_LABEL_FS    = 10
SEC_TICK_FS     = 9
SEC_CB_TICK_FS  = 9
SEC_CB_UNIT_FS  = 9
SEC_QK_FS       = 9

# =====================================================================
# 1. Canvas
# =====================================================================
MM    = 1 / 25.4
FIG_W = 183 * MM
FIG_H = 8.2

fig = plt.figure(figsize=(FIG_W, FIG_H))

# =====================================================================
# 2. GridSpec layout
# =====================================================================
outer = GridSpec(
    2, 1,
    figure=fig,
    height_ratios=[2.8, 5.0],
    hspace=0.48,
    left=0.05,
    right=0.97,
    top=0.95,
    bottom=0.07,
)

gs_top = GridSpecFromSubplotSpec(
    1, 3,
    subplot_spec=outer[0],
    wspace=0.06,
)

gs_bot = GridSpecFromSubplotSpec(
    2, 2,
    subplot_spec=outer[1],
    wspace=0.14,
    hspace=0.38,
)

# =====================================================================
# 3. Clip helper
# =====================================================================
def clip_to_extent(ax, extent, crs, n=260):
    lon0, lon1, lat0, lat1 = extent
    lons = np.linspace(lon0, lon1, n)
    lats = np.linspace(lat0, lat1, n)
    ring = np.vstack([
        np.column_stack([lons,                     np.full_like(lons, lat0)]),
        np.column_stack([np.full_like(lats, lon1),  lats                   ]),
        np.column_stack([lons[::-1],                np.full_like(lons, lat1)]),
        np.column_stack([np.full_like(lats, lon0),  lats[::-1]             ]),
    ])
    ax.set_boundary(Path(ring), transform=crs)

# =====================================================================
# 4. Parameters
# =====================================================================
MAP_PROJ   = ccrs.LambertConformal(central_longitude=-41.5, central_latitude=71)
DATA_PROJ  = ccrs.PlateCarree()
EXTENT     = [-75, -10, 58.0, 85.0]
MAP_TITLES = ["Z850: 1450m", "Z850: 1430m", "Z850 Anomaly"]

NC_FILE   = r"C:\Users\11823\Downloads\VUWZTmulti.nc"
CSV_PATH  = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\2025678allfeature.csv"
LATITUDES = [75, 70]
X_STEP    = 13
U_NORM    = 1.0
W_NORM    = -150.0
Q_SCALE   = 22

# =====================================================================
# 5. Load section data
# =====================================================================
ds     = xr.open_dataset(NC_FILE)
T_DIM  = 'valid_time' if 'valid_time' in ds.dims else 'time'
Z_DIM  = 'pressure_level' if 'pressure_level' in ds.dims else 'level'
m1_v   = ds.sel({T_DIM: slice('1950', '1990')}).mean(T_DIM)
m2_v   = ds.sel({T_DIM: slice('2001', '2024')}).mean(T_DIM)
LONS   = ds['longitude'].values
LEVELS = ds[Z_DIM].values

df = pd.read_csv(CSV_PATH)
topo_funcs = {
    lat: interp1d(
        df[(df['latitude'] >= lat - 0.5) & (df['latitude'] <= lat + 0.5)]
           .groupby('longitude')['altitude'].mean().index,
        df[(df['latitude'] >= lat - 0.5) & (df['latitude'] <= lat + 0.5)]
           .groupby('longitude')['altitude'].mean().values,
        bounds_error=False, fill_value="extrapolate",
    )
    for lat in LATITUDES
}

# =====================================================================
# 6. Map data  ← replace with your actual loaders
# =====================================================================
# m1 = xr.open_dataset("...").sel(...)   # 850 hPa GPH, 1950-1990 mean
# m2 = xr.open_dataset("...").sel(...)   # 850 hPa GPH, 2001-2024 mean
# w1 = xr.open_dataset("...").sel(...)   # surface wind, 1950-1990 mean
# w2 = xr.open_dataset("...").sel(...)   # surface wind, 2001-2024 mean

vmin_m, vmax_m = 1375.0, 1475.0
N_LEVELS_FILL  = 13   # discrete levels for contourf

m_diff   = m2 - m1
w_diff   = w2 - w1
diff_max = float(np.nanmax(np.abs(m_diff['z'].values)))

# =====================================================================
# 7. Draw ABC row
# =====================================================================
map_axes = []
for col, label in enumerate(['a', 'b', 'c']):
    ax = fig.add_subplot(gs_top[0, col], projection=MAP_PROJ)
    ax.set_extent(EXTENT, crs=DATA_PROJ)
    clip_to_extent(ax, EXTENT, DATA_PROJ)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6,
                   edgecolor='grey', zorder=10)

    ax.text(-0.03, 1.06, label,
            transform=ax.transAxes,
            fontsize=PANEL_FONTSIZE, fontweight='bold', va='top')

    ax.set_title(MAP_TITLES[col], fontsize=ABC_TITLE_FS,
                 fontweight='bold', pad=6)
    map_axes.append(ax)

# ── Latitude tick marks on panel a ───────────────────────────────────
ax_a = map_axes[0]
lon_edge = EXTENT[0]
TICK_LEN = 0.022
TICK_PAD = 0.010
D_LAT    = 0.6

for lat_tick in [60, 70, 80]:
    xd, yd = MAP_PROJ.transform_point(lon_edge, lat_tick, DATA_PROJ)
    p_anc  = np.array(ax_a.transAxes.inverted().transform(
                 ax_a.transData.transform((xd, yd))))

    xd_lo, yd_lo = MAP_PROJ.transform_point(lon_edge, lat_tick - D_LAT, DATA_PROJ)
    xd_hi, yd_hi = MAP_PROJ.transform_point(lon_edge, lat_tick + D_LAT, DATA_PROJ)
    p_lo = np.array(ax_a.transAxes.inverted().transform(
               ax_a.transData.transform((xd_lo, yd_lo))))
    p_hi = np.array(ax_a.transAxes.inverted().transform(
               ax_a.transData.transform((xd_hi, yd_hi))))

    merid = p_hi - p_lo
    m_len = np.hypot(*merid)
    if m_len == 0:
        continue
    merid /= m_len

    outward = np.array([merid[1], -merid[0]])
    if outward[0] > 0:
        outward = -outward

    if not (-0.08 <= p_anc[1] <= 1.08):
        continue

    p_tip = p_anc + outward * TICK_LEN
    ax_a.plot(
        [p_anc[0], p_tip[0]], [p_anc[1], p_tip[1]],
        transform=ax_a.transAxes,
        color='black', lw=0.9, clip_on=False, zorder=20,
    )

    p_lbl = p_tip + outward * TICK_PAD
    ax_a.text(
        p_lbl[0], p_lbl[1],
        f"{lat_tick}°N",
        transform=ax_a.transAxes,
        fontsize=SEC_TICK_FS, fontweight='bold',
        va='center', ha='right',
        clip_on=False,
    )

# ── panels a & b: contourf fill (alpha=0.5) + contour lines + quivers ─
wind_datasets = [w1, w2]
cf_ab = None  # keep last reference for colorbar

for i, lvl in enumerate([1450, 1430]):
    # alpha=0.5: half-saturation, clean discrete fill without water-texture artifacts
    cf_ab = map_axes[i].contourf(
        m1.longitude, m1.latitude, m1['z'],
        levels=np.linspace(vmin_m, vmax_m, N_LEVELS_FILL),
        cmap='RdBu_r',
        alpha=0.5,
        transform=DATA_PROJ,
        extend='both',
        zorder=1,
    )
    map_axes[i].contour(
        m1.longitude, m1.latitude, m1['z'],
        levels=[lvl], colors='blue', linewidths=1.2,
        transform=DATA_PROJ, zorder=5,
    )
    map_axes[i].contour(
        m2.longitude, m2.latitude, m2['z'],
        levels=[lvl], colors='red', linewidths=1.2,
        linestyles='--', transform=DATA_PROJ, zorder=5,
    )
    w_panel = wind_datasets[i]
    q_ds = w_panel.isel(latitude=slice(None, None, 20),
                        longitude=slice(None, None, 25))
    q_ref = map_axes[i].quiver(
        q_ds.longitude.values, q_ds.latitude.values,
        q_ds['u10'].values, q_ds['v10'].values,
        transform=DATA_PROJ,
        color='black',
        scale=55,
        width=0.003,
        alpha=1,
        zorder=8,
    )

# ── panel c: contourf fill (alpha=0.5) + contour lines ───────────────
cf_c = map_axes[2].contourf(
    m_diff.longitude, m_diff.latitude, m_diff['z'],
    levels=np.linspace(-diff_max, diff_max, N_LEVELS_FILL),
    cmap='RdBu_r',
    alpha=0.5,
    transform=DATA_PROJ,
    extend='both',
    zorder=1,
)
cs_p = map_axes[2].contour(
    m_diff.longitude, m_diff.latitude, m_diff['z'],
    levels=np.arange(2, 22, 2), colors='red',
    linewidths=0.8, transform=DATA_PROJ, zorder=5,
)
cs_n = map_axes[2].contour(
    m_diff.longitude, m_diff.latitude, m_diff['z'],
    levels=np.arange(-20, 0, 2), colors='blue',
    linewidths=0.8, transform=DATA_PROJ, zorder=5,
)
map_axes[2].clabel(cs_p, inline=True, fontsize=7, fmt='%+.0f')
map_axes[2].clabel(cs_n, inline=True, fontsize=7, fmt='%+.0f')

# ── panel c wind anomaly quiver — commented out, restore when needed ──
# qd_ds = w_diff.isel(latitude=slice(None, None, 20),
#                     longitude=slice(None, None, 25))
# q_anom = map_axes[2].quiver(
#     qd_ds.longitude.values, qd_ds.latitude.values,
#     qd_ds['u10'].values, qd_ds['v10'].values,
#     transform=DATA_PROJ,
#     color='black',
#     scale=55,
#     width=0.003,
#     alpha=1,
#     zorder=8,
# )

# ── ABC colorbars ─────────────────────────────────────────────────────
top_bbox = outer[0].get_position(fig)

cax1_y0 = top_bbox.y0 - 0.068
cax1_h  = 0.013

cax2_x0 = top_bbox.x0 + (top_bbox.x1 - top_bbox.x0) * 0.66
cax2_w  = (top_bbox.x1 - top_bbox.x0) * 0.32

cax1_w  = cax2_w
cax1_x0 = top_bbox.x0 + 0.02

cax1 = fig.add_axes([cax1_x0, cax1_y0, cax1_w, cax1_h])
cb1  = fig.colorbar(cf_ab, cax=cax1, orientation='horizontal')
cb1.set_label("850 hPa GPH (m)", fontsize=ABC_CB_LABEL_FS,
              fontweight='bold', labelpad=1)
cb1.set_ticks([1375, 1425, 1475])
cb1.ax.tick_params(labelsize=ABC_CB_TICK_FS, width=0.5, length=2)
for lbl in cb1.ax.get_xticklabels():
    lbl.set_fontweight('bold')
cb1.outline.set_linewidth(0.5)

cax2 = fig.add_axes([cax2_x0, cax1_y0, cax2_w, cax1_h])
cb2  = fig.colorbar(cf_c, cax=cax2, orientation='horizontal')
cb2.set_label("$\\Delta$ GPH (m)", fontsize=ABC_CB_LABEL_FS,
              fontweight='bold', labelpad=1)
cb2.set_ticks([-diff_max, 0, diff_max])
cb2.ax.tick_params(labelsize=ABC_CB_TICK_FS, width=0.5, length=2)
for lbl in cb2.ax.get_xticklabels():
    lbl.set_fontweight('bold')
cb2.outline.set_linewidth(0.5)

# ── ABC legend & quiverkeys ───────────────────────────────────────────
SHIFT_RIGHT = 0.12

leg_el = [
    Line2D([0], [0], color='blue', lw=1.5, label='1950–1990'),
    Line2D([0], [0], color='red',  lw=1.5, ls='--', label='2001–2024'),
]
fig.legend(
    handles=leg_el,
    loc='lower left',
    ncol=2,
    frameon=False,
    prop={'weight': 'bold', 'size': ABC_LEG_FS},
    bbox_to_anchor=(top_bbox.x0 + 0.01 + SHIFT_RIGHT, cax1_y0 + 0.022),
    bbox_transform=fig.transFigure,
)
plt.quiverkey(
    q_ref,
    X=top_bbox.x0 - 0.05 + SHIFT_RIGHT - 0.03, Y=cax1_y0 + 0.048,
    U=5, label="5 m s$^{-1}$", labelpos='E',
    coordinates='figure',
    fontproperties={'size': ABC_LEG_FS, 'weight': 'bold'},
)
# panel c quiverkey — commented out together with q_anom above
# plt.quiverkey(
#     q_anom,
#     X=cax2_x0 + SHIFT_RIGHT, Y=cax1_y0 + 0.048,
#     U=1,
#     label="1 m s$^{-1}$",
#     labelpos='E',
#     coordinates='figure',
#     fontproperties={'size': ABC_LEG_FS, 'weight': 'bold'},
# )

# =====================================================================
# 8. Colorbar tick rounding helper
# =====================================================================
def round_to_clean(val):
    """
    Round vmax UP to a clean number:
      val >= 1  → nearest integer
      val <  1  → nearest 0.1
    """
    if val == 0:
        return 0.0
    if val >= 1:
        return float(math.ceil(val))
    else:
        return float(math.ceil(val * 10)) / 10.0

def fmt_for_vmax(vmax):
    return '%.0f' if vmax >= 1 else '%.1f'

# =====================================================================
# 9. Draw DEFG row  (75°N and 70°N)
# =====================================================================
panel_labels_bot = ['d', 'e', 'f', 'g']
units_bot        = ["(K)", "(Pa s$^{-1}$)"]
col_titles_bot   = ["T anomaly", "Vertical velocity anomaly"]

Q_last = None

def clip_percentile(arr, pct=95):
    valid = arr.compressed() if np.ma.is_masked(arr) else arr.ravel()
    valid = valid[~np.isnan(valid)]
    if len(valid) == 0:
        return arr
    threshold = np.percentile(np.abs(valid), pct)
    return np.ma.clip(arr, -threshold, threshold)

for i, lat in enumerate(LATITUDES):
    ta = (m2_v['t'] - m1_v['t']).sel(latitude=lat, method='nearest').transpose(Z_DIM, 'longitude')
    wa = (m2_v['w'] - m1_v['w']).sel(latitude=lat, method='nearest').transpose(Z_DIM, 'longitude')
    ua = (m2_v['u'] - m1_v['u']).sel(latitude=lat, method='nearest').transpose(Z_DIM, 'longitude')

    topo_hPa = 1013.25 * (1 - 2.25577e-5 * topo_funcs[lat](LONS)) ** 5.25588

    lons_sub = LONS[::X_STEP]
    levs_sub = LEVELS[::2]
    Xg, Yg  = np.meshgrid(lons_sub, levs_sub)
    U_plot  = ua.values[::2, ::X_STEP] * U_NORM
    W_plot  = wa.values[::2, ::X_STEP] * W_NORM

    mask   = Yg > (1013.25 * (1 - 2.25577e-5 * topo_funcs[lat](Xg)) ** 5.25588 + 20)
    U_plot = np.ma.masked_where(mask, U_plot)
    W_plot = np.ma.masked_where(mask, W_plot)
    U_plot = clip_percentile(U_plot, 95)
    W_plot = clip_percentile(W_plot, 95)

    for j, data in enumerate([ta, wa]):
        ax = fig.add_subplot(gs_bot[i, j])
        vmax_raw = np.nanmax(np.abs(data.values))

        vmax    = round_to_clean(vmax_raw)
        fmt_str = fmt_for_vmax(vmax)

        cf = ax.contourf(
            LONS, LEVELS, data,
            levels=np.linspace(-vmax, vmax, 21),
            cmap=plt.cm.RdBu_r, extend='both',
        )

        cbar = plt.colorbar(cf, ax=ax, shrink=0.85,
                            format=ticker.FormatStrFormatter(fmt_str))
        cbar.set_ticks([-vmax, 0, vmax])
        cbar.ax.tick_params(width=0.5, length=2, labelsize=SEC_CB_TICK_FS)
        for lbl in cbar.ax.get_yticklabels():
            lbl.set_fontweight('bold')
        cbar.outline.set_linewidth(0.5)

        # Unit annotation — first row only (75°N)
        if i == 0:
            cbar.ax.text(
                2.2, -0.3, units_bot[j],
                fontsize=SEC_CB_UNIT_FS, fontweight='bold',
                transform=cbar.ax.transAxes,
                ha='left', va='center',
            )

        # Right column: quivers
        if j == 1:
            Q = ax.quiver(
                Xg, Yg, U_plot, W_plot,
                scale=Q_SCALE * 0.833,
                width=0.0035 * 1.2,
                headwidth=3, headlength=6, headaxislength=5.5,
                color='black', pivot='middle',
            )
            Q_last = Q

        # Topography
        ax.fill_between(LONS, topo_hPa, 1100, color='0.8', zorder=10)
        ax.plot(LONS, topo_hPa, color='black', lw=0.5, zorder=11)

        ax.set_ylim(1000, 200)
        ax.set_xlim(-75, -10)
        ax.set_yticks(np.linspace(1000, 200, 5))

        ax.tick_params(direction='out', width=0.6, length=3,
                       labelsize=SEC_TICK_FS)

        if i < 1:
            ax.set_xticklabels([])
        else:
            ax.set_xticks([-60, -40, -20])
            ax.set_xticklabels(
                ['60°W', '40°W', '20°W'],
                fontsize=SEC_TICK_FS, fontweight='bold',
            )

        if j == 0:
            ax.set_ylabel(f"{lat}°N\nPressure (hPa)",
                          fontsize=SEC_LABEL_FS, fontweight='bold')

        if i == 1:
            ax.set_xlabel("Longitude",
                          fontsize=SEC_LABEL_FS, fontweight='bold')

        ax.text(
            -0.01, 1.06,
            panel_labels_bot[i * 2 + j],
            transform=ax.transAxes,
            fontsize=PANEL_FONTSIZE, fontweight='bold',
            va='bottom', ha='left',
        )

        if i == 0:
            ax.text(
                0.5, 1.06,
                col_titles_bot[j],
                transform=ax.transAxes,
                fontsize=SEC_TITLE_FS, fontweight='bold',
                va='bottom', ha='center',
            )

# ── Section quiverkey ────────────────────────────────────────────────
if Q_last is not None:
    plt.quiverkey(
        Q_last,
        X=0.88, Y=0.04,
        U=3.6,
        label="3 m s$^{-1}$",
        labelpos='E',
        coordinates='figure',
        fontproperties={'size': SEC_QK_FS, 'weight': 'bold'},
    )

# =====================================================================
# 10. Save
# =====================================================================
OUT_DIR = r"C:\Users\11823\era5land"
plt.savefig(OUT_DIR + r"\Fig3.pdf", bbox_inches='tight')
plt.savefig(OUT_DIR + r"\Fig3.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved to", OUT_DIR)

图7  站点验证

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from scipy.stats import pearsonr, gaussian_kde
import xgboost as xgb
import lightgbm as lgb
import catboost

warnings.filterwarnings("ignore")

# ============================================================
# 1. NPJ Style & Fonts
# ============================================================
PT_PANEL_LABEL = 14
PT_COL_TITLE   = 9
PT_AXIS_LABEL  = 9
PT_TICK        = 8
PT_STATS       = 7
PT_CBAR_TICK   = 8
PT_ROW_TITLE   = 8

rcParams.update({
    "font.family"       : "Arial",
    "axes.linewidth"    : 0.6,
    "axes.labelweight"  : "bold",
    "figure.dpi"        : 300,
})

def metrics_r_rmse_mae(y_true, y_pred):
    yt, yp = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]
    r = pearsonr(yt, yp)[0] if len(yt) > 2 else 0
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = np.mean(np.abs(yp - yt))
    return r, rmse, mae, len(yt)

# 自定义淡色方案
white_green_cmap = LinearSegmentedColormap.from_list("pale_green", ["#FFFFFF", "#98FB98"])
cmaps = {"R": "Reds", "RMSE": white_green_cmap, "MAE": "Blues"}

# ============================================================
# 2. Paths, Maps & ML Configuration
# ============================================================
# Map CSVs
SAMPLE_CSV = r"C:\Users\greenlandProject\ERA5LAND\siteaccuracy_sample.csv"
YEAR_CSV   = r"C:\Users\greenlandProject\ERA5LAND\siteaccuracy_year.csv"
LOSO_CSV   = r"C:\Users\greenlandProject\ERA5LAND\siteaccuracy_loso.csv"

# ML Data
FILE_PATH_MAIN = r"C:\Users\greenlandProject\ERA5LAND\merged_allfeature.csv"
FILE_PATH_TIME = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1_with_pred_917.xlsx"

# Shapefiles
BASIN_DIR    = r"C:\greenland\Greenland_shp\流域冰盖"
CONTOUR_PATH = r"C:\greenland\Greenland_shp\等间距等值线.shp"

# 流域名称映射 (备用)
BASIN_NAME_MAP = {
    "northwest": "NW", "southwest": "SW", "northeast": "NE", "southeast": "SE",
    "westnorth": "NW", "westsouth": "SW", "eastnorth": "NE", "eastsouth": "SE",
    "northwes":  "NW", "southwes":  "SW", "northeas":  "NE", "southeas":  "SE",
    "west": "W", "east": "E", "north": "N", "south": "S", "middle": "M",
    "nw": "NW", "sw": "SW", "ne": "NE", "se": "SE",
    "w": "W",   "e": "E",   "n": "N",   "s": "S",   "m": "M"
}
VALID_ABBR = {"NW", "NE", "SW", "SE", "W", "E", "N", "S"}

TARGET = "albedo"
STATION_LOSO = "TAS_L"
INPUT_FEATURES = ["longitude", "latitude", "altitude_last_valid", "sp", "ssrd", "str", "strd",
                  "t2m", "u10", "v10", "snowc", "sde", "asn", "year", "month"]
EXCLUDE_STATIONS = ["QAS_M", "MIT", "NUK_K"]

models = {
    'RF': RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    'XGB': xgb.XGBRegressor(n_estimators=300, n_jobs=-1, random_state=42),
    'GBDT': GradientBoostingRegressor(n_estimators=300, random_state=42),
    'LGBM': lgb.LGBMRegressor(n_estimators=400, n_jobs=-1, random_state=42, verbose=-1),
    'CatBoost': catboost.CatBoostRegressor(iterations=500, verbose=0, random_seed=42)
}

def fit_predict_ensemble(X_train, y_train, X_test):
    preds = [clone(m).fit(X_train, y_train).predict(X_test) for m in models.values()]
    return np.mean(np.column_stack(preds), axis=1)

# ============================================================
# 3. Data Processing & Machine Learning Pipeline
# ============================================================
print(">>> Loading Map CSVs...")
row_dfs = []
for path in [SAMPLE_CSV, YEAR_CSV, LOSO_CSV]:
    # 为了防止文件不存在报错，这里加个简单判断（如果在真实环境中跑，请确保路径正确）
    if os.path.exists(path):
        df = pd.read_csv(path)
        if "Bias_abs" in df.columns:
            df = df.rename(columns={"Bias_abs": "MAE"})
        row_dfs.append(df)
    else:
        # 仅为演示时不报错的占位数据处理
        row_dfs.append(pd.DataFrame({"longitude": [], "latitude": [], "R": [], "RMSE": [], "MAE": []}))

print(">>> Loading main CSV for ML...")
if os.path.exists(FILE_PATH_MAIN):
    data = pd.read_csv(FILE_PATH_MAIN)
    data = data[~data["station"].isin(EXCLUDE_STATIONS)].copy()
    mask = data[TARGET].notna() & ~data[INPUT_FEATURES].isna().any(axis=1)
    data_clean = data.loc[mask].copy()
    X = data_clean[INPUT_FEATURES].astype(float)
    y = data_clean[TARGET].astype(float)
    stations = data_clean["station"].astype(str)

    print(">>> Training Sample-based model...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    y_te = y_test.values
    y_pred_sample = fit_predict_ensemble(X_train, y_train, X_test)

    print(f">>> Training Site-based LOSO for {STATION_LOSO}...")
    train_mask = stations != STATION_LOSO
    test_mask = stations == STATION_LOSO
    y_pred_site = fit_predict_ensemble(X[train_mask], y[train_mask], X[test_mask])
    y_true_site = y[test_mask].values
else:
    # 仅为代码顺利走通的占位
    y_te = y_pred_sample = y_true_site = y_pred_site = np.random.rand(100)

print(">>> Loading Excel for time-based 2025...")
if os.path.exists(FILE_PATH_TIME):
    df_time = pd.read_excel(FILE_PATH_TIME)
    df_2025 = df_time[df_time["year"] == 2025].copy()
    df_2025 = df_2025.replace([np.inf, -np.inf], np.nan).dropna(subset=["st_albedo", "albedo_pred"])
    y_true_time = df_2025["st_albedo"].astype(float).values
    y_pred_time = df_2025["albedo_pred"].astype(float).values
else:
    y_true_time = y_pred_time = np.random.rand(100)

# ============================================================
# 4. Figure Setup & GridSpec
# ============================================================
print(">>> Rendering Figure...")

# 在缩小 10% (0.9) 的基础上，再次缩小 10% -> 0.9 * 0.9 = 0.81
FIG_WIDTH_IN = (183 / 25.4) * 0.81
FIG_HEIGHT_IN = (183 / 25.4) * 1.55 * 0.81
fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

# d 图散点网格：保持上一次更新的紧凑两边留白
gs_sc  = fig.add_gridspec(1, 3, left=0.08, right=0.97, bottom=0.06, top=0.23, wspace=0.25)

# abc 图地图网格：保持拓宽后的边界
MAP_LEFT  = 0.11
MAP_RIGHT = 0.98
gs_map = fig.add_gridspec(3, 3, left=MAP_LEFT, right=MAP_RIGHT,
                           top=0.96, bottom=0.32, wspace=-0.15, hspace=0.12)

# 地图投影和基础图层
PROJ   = ccrs.NorthPolarStereo(central_longitude=-45)
PC     = ccrs.PlateCarree()
EXTENT = [-57, -19, 59.5, 83.5]

land_feature = cfeature.NaturalEarthFeature("physical", "land", "50m")
greenland_main = sorted([g for g in list(land_feature.geometries()) if g.bounds[0]>-75 and g.bounds[1]>58], key=lambda x: x.area, reverse=True)[:1]

# 预加载流域 Shapefile 数据
print(">>> Loading Basin Shapefiles...")
basin_geometries = []
if os.path.isdir(BASIN_DIR):
    for shp_file in glob.glob(os.path.join(BASIN_DIR, "*.shp")):
        reader = shpreader.Reader(shp_file)
        basin_geometries.extend(list(reader.geometries()))
elif os.path.isfile(BASIN_DIR) or os.path.isfile(BASIN_DIR + ".shp"):
    # 兼容路径直接指向某个具体 shp 文件的写法
    target_shp = BASIN_DIR if BASIN_DIR.endswith('.shp') else BASIN_DIR + '.shp'
    if os.path.exists(target_shp):
        reader = shpreader.Reader(target_shp)
        basin_geometries.extend(list(reader.geometries()))

# ============================================================
# 4.1 A/B/C: Map Panels
# ============================================================
axes = np.empty((3, 3), dtype=object)
scatters = {}

row_letters = ['a', 'b', 'c']
row_side_titles = ['Sample test', 'Year test\n(LOYO)', 'Station test\n(LOSO)']
col_titles = ["Pearson's r", "RMSE", "MAE"]
metrics_list = ["R", "RMSE", "MAE"]

for r_i, df in enumerate(row_dfs):
    for c_i, m in enumerate(metrics_list):
        ax = fig.add_subplot(gs_map[r_i, c_i], projection=PROJ)
        axes[r_i, c_i] = ax
        ax.set_extent(EXTENT, PC)
        
        # 绘制格陵兰底图
        ax.add_geometries(greenland_main, crs=PC, facecolor="#F5F5F5", edgecolor="black", linewidth=0.4, alpha=0.9, zorder=1)
        
        # 追加流域细分界线 (灰色细线，不加标注)
        if basin_geometries:
            ax.add_geometries(basin_geometries, crs=PC, facecolor='none', edgecolor='gray', linewidth=0.3, alpha=0.8, zorder=2)
        
        # 散点图
        if not df.empty:
            sc = ax.scatter(df["longitude"], df["latitude"], c=df[m], cmap=cmaps[m], s=8,
                            edgecolors="black", linewidth=0.15, transform=PC, zorder=5,
                            vmin=0, vmax=(0.2 if m != "R" else 1.0))
            if r_i == 2: scatters[m] = sc
            
        ax.set_aspect("auto")
        ax.axis("off")

# ── 行面板字母 & 行侧标题 ──
X_LEFT  = 0.01   
X_TITLE = 0.04   
for r_i in range(3):
    p = axes[r_i, 0].get_position()
    fig.text(X_LEFT, p.y1, row_letters[r_i], fontsize=PT_PANEL_LABEL, fontweight="bold")
    fig.text(X_TITLE, (p.y0+p.y1)/2, row_side_titles[r_i],
             ha="center", va="center", rotation=90, fontsize=PT_ROW_TITLE, fontweight="bold")

# ── 列标题 ──
ORIG_COL_CENTERS = [0.255 - 0.025, 0.545 - 0.025, 0.835 - 0.025]
for c_i, title in enumerate(col_titles):
    p = axes[0, c_i].get_position()
    fig.text(ORIG_COL_CENTERS[c_i], p.y1 + 0.005, title,
             ha="center", va="bottom", fontsize=PT_COL_TITLE, fontweight="bold")

# ── 色标 ──
for c_i, m in enumerate(metrics_list):
    if m in scatters:
        p = axes[2, c_i].get_position()   
        cw, ch = p.width * 0.7, 0.007
        cax = fig.add_axes([p.x0 + (p.width - cw) / 2 - 0.03, 0.285, cw, ch])
        cb = fig.colorbar(scatters[m], cax=cax, orientation="horizontal")
        cb.set_ticks([0, 0.5, 1] if m == "R" else [0, 0.1, 0.2])
        cb.ax.tick_params(labelsize=PT_CBAR_TICK - 1, pad=1, length=2)
        for l in cb.ax.get_xticklabels(): l.set_fontweight('bold')
        cb.outline.set_linewidth(0.5)

# ============================================================
# 4.2 D: Scatter Validation Panels
# ============================================================
scatter_configs = [
    (y_te, y_pred_sample, "Sample-based"),
    (y_true_site, y_pred_site, "Site-based"),
    (y_true_time, y_pred_time, "Time-based")
]

for i, (yt, yp, title) in enumerate(scatter_configs):
    ax = fig.add_subplot(gs_sc[0, i])
    ax.set_box_aspect(1)
    
    msk = np.isfinite(yt) & np.isfinite(yp)
    yt_c, yp_c = np.array(yt)[msk], np.array(yp)[msk]
    
    if len(yt_c) > 3000:
        idx = np.random.choice(len(yt_c), 3000, replace=False)
        yt_sub, yp_sub = yt_c[idx], yp_c[idx]
    else:
        yt_sub, yp_sub = yt_c, yp_c

    if len(yt_sub) > 0:
        xy = np.vstack([yt_sub, yp_sub])
        z = gaussian_kde(xy)(xy)
        ax.scatter(yt_sub, yp_sub, c=z, s=4, cmap="viridis", alpha=0.7, edgecolor="none")
    
    ax.plot([0, 1], [0, 1], "k--", lw=0.7, alpha=0.4)
    
    r_val, rmse_val, mae_val, n_val = metrics_r_rmse_mae(yt_c, yp_c)
    stats_txt = f"r: {r_val:.3f}\nRMSE: {rmse_val:.3f}\nMAE: {mae_val:.3f}\nN: {n_val:,}"
    ax.text(0.95, 0.05, stats_txt, transform=ax.transAxes, ha="right", va="bottom", 
            fontsize=PT_STATS, fontweight="bold")
    
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks([0.5, 1.0]) 
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    
    ax.set_title(title, fontsize=PT_COL_TITLE, fontweight="bold", pad=6)
    
    # 改回 In situ albedo
    ax.set_xlabel("In situ albedo", fontweight="bold", fontsize=PT_AXIS_LABEL)   
    if i == 0: ax.set_ylabel("Reconstruct albedo", fontweight="bold", fontsize=PT_AXIS_LABEL)
    ax.tick_params(direction="in", labelsize=PT_TICK)
    for l in (ax.get_xticklabels() + ax.get_yticklabels()): l.set_fontweight('bold')

fig.text(X_LEFT, 0.235, "d", fontsize=PT_PANEL_LABEL, fontweight="bold")

# ============================================================
# 5. Final Export (已注释，仅显示)
# ============================================================
OUT_DIR = r"C:\Users\11823\era5land"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_FIG_BASE = os.path.join(OUT_DIR, "Fig7")
for ext in ['png', 'pdf']:
 save_path = f"{OUT_FIG_BASE}.{ext}"
fig.savefig(save_path, dpi=300, bbox_inches="tight")

print("✅ Completed: Rendering finished (Not saved to disk).")
plt.show()

训练模型和保存pkl等文件，预备后期生成

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split, KFold
import xgboost as xgb
import lightgbm as lgb
import catboost
import shap
import matplotlib.pyplot as plt
import joblib
import os
from datetime import datetime

# ========== 1. 读取数据 ==========
file_path = r"C:\Users\greenlandProject\ERA5LAND\merged_allfeature.csv"
input_features = [
    "longitude", "latitude", "altitude_last_valid", "sp", "ssrd", "str", "strd",
    "t2m", "u10", "v10", "snowc", "sde", "asn", "year", "month"
]
target_column = "albedo"
exclude_stations = ["QAS_M", "MIT", "NUK_K"]

data = pd.read_csv(file_path)
data = data[~data["station"].isin(exclude_stations)]

X = data[input_features].astype(float)
y = data[target_column]

# 检查缺失特征
missing_features = set(input_features) - set(data.columns)
if missing_features:
    raise ValueError(f"Missing features in data: {missing_features}")

# 拆分训练 / 测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ========== 2. 定义基础模型 ==========
models = {
    'RF': RandomForestRegressor(n_jobs=-1, random_state=42),
    'XGB': xgb.XGBRegressor(n_jobs=-1, random_state=42),
    'GBDT': GradientBoostingRegressor(random_state=42),
    'LGBM': lgb.LGBMRegressor(n_jobs=-1, random_state=42),
    'CatBoost': catboost.CatBoostRegressor(verbose=0, random_seed=42)
}

# ========== 3. 构造元特征 (K折) ==========
kf = KFold(n_splits=5, shuffle=True, random_state=42)
meta_features = np.zeros((X_train.shape[0], len(models)))
model_statistics = {}

for idx, (name, model) in enumerate(models.items()):
    meta_feature = np.zeros(X_train.shape[0])

    for train_idx, val_idx in kf.split(X_train):
        X_t, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_t, y_t)
        preds = model.predict(X_val)
        meta_feature[val_idx] = preds

    meta_features[:, idx] = meta_feature

    # CV指标
    r2 = r2_score(y_train, meta_feature)
    r, _ = pearsonr(y_train, meta_feature)
    mse = mean_squared_error(y_train, meta_feature)
    rmse = np.sqrt(mse)
    bias = np.mean(meta_feature - y_train)
    model_statistics[name] = {"R2": r2, "R": r, "RMSE": rmse, "MSE": mse, "Bias": bias}

# ========== 4. 训练元模型 ==========
meta_model = Ridge()
meta_model.fit(meta_features, y_train)

# ========== 5. 训练最终基础模型 ==========
trained_base_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_base_models[name] = model

# ========== 6. 测试集预测 ==========
test_meta_features = np.column_stack([
    trained_base_models[name].predict(X_test) for name in models
])
final_preds = meta_model.predict(test_meta_features)

# ========== 7. SHAP分析 ==========
shap_model = trained_base_models['RF']  # 用RF解释
try:
    explainer = shap.TreeExplainer(shap_model)
    shap_values = explainer.shap_values(X_test)

    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title("SHAP Summary (RF)")
    plt.tight_layout()
    plt.savefig("shap_summary_917.png")
    plt.close()

    plt.figure()
    shap.plots.bar(explainer(X_test), max_display=15, show=False)
    plt.title("Top 15 Features by SHAP Value")
    plt.tight_layout()
    plt.savefig("shap_bar_917.png")
    plt.close()
except Exception as e:
    print(f"⚠️ SHAP Error: {str(e)}")

# ========== 8. 输出性能 ==========
r2 = r2_score(y_test, final_preds)
r, _ = pearsonr(y_test, final_preds)
mse = mean_squared_error(y_test, final_preds)
rmse = np.sqrt(mse)
bias = np.mean(final_preds - y_test)

print("\n========== Final Stacking Model Test Evaluation ==========")
print(f"R²: {r2:.6f}")
print(f"Pearson R: {r:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"Bias: {bias:.6f}")

print("\n========== Base Model Cross-Validation Metrics ==========")
for name, stats in model_statistics.items():
    print(f"\n{name.upper()}:")
    for metric, value in stats.items():
        print(f"{metric}: {value:.6f}")

print("\n✅ 实际使用的特征:", list(X_train.columns))

# ========== 9. 保存模型 ==========
save_dir = "saved_models_917"
os.makedirs(save_dir, exist_ok=True)

# 保存基础模型
for name, model in trained_base_models.items():
    joblib.dump(model, os.path.join(save_dir, f"{name}_917.pkl"))

# 保存元模型
joblib.dump(meta_model, os.path.join(save_dir, "meta_model_917.pkl"))

print(f"\n✅ 模型已保存到 {save_dir}")


用pkl文件完成所有年月的albedo预测，生成月为单位的csv文件

In [ ]:
import joblib
import pandas as pd
import numpy as np
import os
import warnings
import re

# 屏蔽一切警告
warnings.filterwarnings("ignore")

# 加载整体堆叠模型
stacking_model_path = r"C:\Users\11823\era5land\202209202402\NoSdestacking_model_Pre202209.pkl"
try:
    stacking_model = joblib.load(stacking_model_path)
    trained_base_models = stacking_model['trained_base_models']
    meta_model = stacking_model['meta_model']
    base_model_names = stacking_model['base_model_names']
    # input_features = stacking_model['input_features'] # 保持使用自定义的 input_features
except FileNotFoundError:
    print(f"❌ 错误：模型文件未找到于 {stacking_model_path}")
    exit()
except KeyError as e:
    print(f"❌ 错误：模型文件中缺少必要的键：{e}")
    exit()

# 输入特征
input_features = [
    "longitude", "latitude", "altitude_last_valid", "sp", "ssrd", "str", "strd",
    "t2m", "u10", "v10", "snowc", "asn", "year", "sde", "month"
]
target_column = "albedo"

# 输入输出路径
input_folder = r"C:\greenland\ProcessedCSV"
output_folder = r"C:\Users\11823\era5land\202209202402"

# 创建输出文件夹（如不存在）
os.makedirs(output_folder, exist_ok=True)

# 设置要处理的年份范围（可修改）
start_year = 1950
end_year = 2024

# 分批处理大小
chunk_size = 1000

# 遍历所有文件
for filename in os.listdir(input_folder):
    # 提取年份（假设文件名中前4位是年份，如199507Greenland_allfeature.csv）
    match = re.match(r"(\d{4})\d{2}.*allfeature\.csv", filename)
    if match:
        year = int(match.group(1))
        
        # 构造预期的输出文件名
        file_prefix = filename.replace("allfeature.csv", "")
        output_file = os.path.join(output_folder, f"{file_prefix}prediction.csv")

        # --- 新增：检查预测结果文件是否已存在，如果存在则跳过 ---
        if os.path.exists(output_file):
            print(f"⏩ 文件 {filename} 的预测结果已存在于 {output_file}，跳过处理。")
            continue
        # --------------------------------------------------------

        if start_year <= year <= end_year:
            filepath = os.path.join(input_folder, filename)
            
            print(f"⏳ 开始处理文件: {filename}")
            
            try:
                # 优化：只读取需要的列，减少内存占用（如果 albedo 不在输入文件中，则不需要读取 target_column）
                # 原始文件中可能包含 target_column，但我们只用 input_features
                data = pd.read_csv(filepath, usecols=input_features) 
                # 重新读取完整的行，以便保留所有原始列和索引
                data_full = pd.read_csv(filepath) 
                data_input = data_full[input_features]


                all_results = []

                # 使用 iloc/loc 进行切片时，需要确保索引的对应性，这里使用完整的 data_full 和 data_input
                for start in range(0, len(data_input), chunk_size):
                    chunk_input = data_input.iloc[start:start + chunk_size]
                    chunk_full = data_full.iloc[start:start + chunk_size]
                    
                    # 找出非NaN的行索引
                    clean_indices = chunk_input.dropna().index
                    
                    # 确保索引非空
                    if clean_indices.empty:
                        continue

                    # 提取干净的数据块
                    chunk_clean_input = chunk_input.loc[clean_indices]
                    
                    # 使用干净的输入进行预测
                    base_preds = [model.predict(chunk_clean_input) for model in trained_base_models]
                    
                    # 检查 base_preds 是否为空，理论上不会如果 clean_indices 不为空
                    if not base_preds:
                        continue
                        
                    stacked_input = np.column_stack(base_preds)
                    final_preds = meta_model.predict(stacked_input)

                    # 在完整的行数据上，根据干净的索引进行切片
                    data_clean = chunk_full.loc[clean_indices].copy()
                    
                    # 将预测结果赋给 albedo 列
                    data_clean[target_column] = final_preds
                    all_results.append(data_clean)

                if all_results:
                    final_data = pd.concat(all_results, axis=0)
                    
                    # 检查 output_file 是否已被检查过（此处冗余，但作为双重保险）
                    if not os.path.exists(output_file):
                        final_data.to_csv(output_file, index=False)
                        print(f"✅ 文件 {output_file} 输出成功，共 {len(final_data)} 行")
                    else:
                        print(f"⚠️ 文件 {filename} 的预测结果已在检查期间生成，不再重复输出。")
                else:
                    print(f"⚠️ 文件 {filename} 没有有效数据，跳过")

            except Exception as e:
                print(f"❌ 处理文件 {filename} 出错：{e}")
                continue

图1

In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from scipy.ndimage import zoom, gaussian_filter
from matplotlib.colors import TwoSlopeNorm, Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ====================== 出版级全局设置 ======================
plt.rcParams.update({
    "font.family":       "Arial",
    "font.size":         10,
    "font.weight":       "bold",
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,
    "xtick.direction":   "in",
    "ytick.direction":   "in",
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.labelsize":    10,
    "legend.fontsize":   9,
    "lines.linewidth":   0.75,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

# ====================== 字号统一常量 ======================
FONT_SIZE         = 14
PANEL_LETTER_SIZE = 16
BASIN_LABEL_SIZE  = 12
CBAR_TICK_SIZE    = 9
TS_FONT           = 10

# ====================== 开关：2001-2024阴影框 ======================
SHOW_SHADE  = True
SHADE_COLOR = 'grey'
SHADE_ALPHA = 0.15

# ====================== 版式参数 ======================
FIG_WIDTH_INCHES  = 6.0
FIG_HEIGHT_INCHES = 10.5

LEFT   = 0.10
RIGHT  = 0.87
TOP    = 0.97
BOTTOM = 0.0

ROW_GAP_MAP = -0.05
ROW_GAP_TS  = -0.01

TOTAL_H = TOP - BOTTOM

TS_FRAC = 0.23
TS_H    = TOTAL_H * TS_FRAC
TS_Y0   = BOTTOM

MAP_Y0 = TS_Y0 + TS_H + ROW_GAP_TS
MAP_H  = TOP - MAP_Y0
ROW_H  = (MAP_H - ROW_GAP_MAP) / 2.0 * 0.75
MAP_USED   = ROW_H * 2 + ROW_GAP_MAP
MAP_OFFSET = 0.0   # 不再偏移，让Mean行顶部贴近TOP
ROW_Y  = [MAP_Y0 + MAP_OFFSET + ROW_H + ROW_GAP_MAP,
          MAP_Y0 + MAP_OFFSET+0.02]

CBAR_SCALE = 0.60
CBAR_W     = 0.018 * CBAR_SCALE
CBAR_X     = 0.885

GAP       = -0.08
SUBPLOT_W = (RIGHT - LEFT - 2 * GAP) / 3.0
COL_X_POS = [
    LEFT+ 0.06,
    LEFT + SUBPLOT_W + GAP+ 0.06,
    LEFT + 2 * (SUBPLOT_W + GAP)+ 0.06,
]

ROW_LABEL_X    = LEFT - 0.035
PANEL_LETTER_X = LEFT - 0.055

TS_LEFT  = LEFT + 0.08
TS_RIGHT = COL_X_POS[2] + SUBPLOT_W
TS_WIDTH = TS_RIGHT - TS_LEFT

TS_INNER_GAP = 0.022
ts_h_upper   = (TS_H - TS_INNER_GAP) * (2 / 3.0)
ts_h_lower   = (TS_H - TS_INNER_GAP) * (1 / 3.0)
ts_y_lower   = TS_Y0
ts_y_upper   = TS_Y0 + ts_h_lower + TS_INNER_GAP

# ====================== 数据与投影路径 ======================
DATA_DIR     = r"C:\greenland\1995bu\fig4"
OUT_DIR      = r"C:\Users\11823\era5land"
BASIN_DIR    = r"C:\greenland\Greenland_shp\流域冰盖"
CONTOUR_PATH = r"C:\greenland\Greenland_shp\等间距等值线.shp"
TS_CSV_PATH  = r"C:\greenland\1995bu\OtherVariables\Fig_Greenland_AllBasin_TimeSeries.csv"

PROJ   = ccrs.NorthPolarStereo(central_longitude=-45)
PC     = ccrs.PlateCarree()
EXTENT = [-57, -19, 59.5, 83.5]

# ====================== 流域名称映射 ======================
BASIN_NAME_MAP = {
    "northwest": "NW", "southwest": "SW", "northeast": "NE", "southeast": "SE",
    "westnorth": "NW", "westsouth": "SW", "eastnorth": "NE", "eastsouth": "SE",
    "northwes":  "NW", "southwes":  "SW", "northeas":  "NE", "southeas":  "SE",
    "west": "W", "east": "E", "north": "N", "south": "S", "middle": "M",
    "nw": "NW", "sw": "SW", "ne": "NE", "se": "SE",
    "w": "W",   "e": "E",   "n": "N",   "s": "S",   "m": "M"
}
VALID_ABBR = {"NW", "NE", "SW", "SE", "W", "E", "N", "S"}

# ====================== 数据处理函数 ======================
def load_and_grid(csv_name):
    path  = os.path.join(DATA_DIR, csv_name)
    df    = pd.read_csv(path, usecols=["lat", "lon", "mean", "trend_per_decade", "trend_sig95"])
    df    = df.dropna(subset=["lat", "lon"]).drop_duplicates(subset=["lat", "lon"])
    mean  = df.pivot(index="lat", columns="lon", values="mean").sort_index(ascending=True)
    trend = df.pivot(index="lat", columns="lon", values="trend_per_decade").sort_index(ascending=True)
    sig   = df.pivot(index="lat", columns="lon", values="trend_sig95").sort_index(ascending=True)
    return mean.index.values, mean.columns.values, mean.values, trend.values, sig.values

def fast_smooth(Z, factor=3, sigma=1):
    mask      = np.isfinite(Z)
    Z_filled  = np.where(mask, Z, 0.0)
    Z_zoom    = zoom(Z_filled,           zoom=factor, order=1, mode="nearest")
    mask_zoom = zoom(mask.astype(float), zoom=factor, order=1, mode="nearest")
    Z_smooth  = gaussian_filter(Z_zoom,    sigma=sigma, mode='nearest')
    mask_sm   = gaussian_filter(mask_zoom, sigma=sigma, mode='nearest')
    return np.divide(Z_smooth, mask_sm, where=mask_sm > 1e-6,
                     out=np.full_like(Z_smooth, np.nan))

# ====================== 绘图辅助函数 ======================
DOT_STEP         = 15
DOT_STEP_LASTCOL = 12
DOT_SIZE         = 10
DOT_ALPHA        = 0.35
DOT_MARKER       = '.'

def draw_basins(ax, basins_gdf):
    if basins_gdf.empty:
        return
    gdf2 = basins_gdf[basins_gdf["abbr"].isin(VALID_ABBR)]
    if gdf2.empty:
        return
    ax.add_geometries(gdf2.geometry, crs=PC,
                      edgecolor="grey", facecolor="none", linewidth=0.5, zorder=6)
    offsets = {
        "NW": (1.0, 0), "SW": (-0.5, 0), "NE": (-1.0, 0), "SE": (1.0, 0),
        "W":  (1.5, 0), "E":  (-0.8, 0), "N":  (0, -0.6), "S":  (0, +0.6)
    }
    for _, row in gdf2.iterrows():
        abbr = row.get("abbr", "")
        if abbr not in VALID_ABBR:
            continue
        pt     = row.geometry.representative_point()
        dx, dy = offsets.get(abbr, (0, 0))
        ax.text(pt.x + dx, pt.y + dy, abbr,
                fontsize=BASIN_LABEL_SIZE, fontweight="bold",
                color="grey", ha="center", transform=PC, zorder=10)

def draw_row_colorbar(fig, y0, row_h, cmap, norm, row_type):
    h   = row_h * 0.78 * CBAR_SCALE
    cax = fig.add_axes([CBAR_X, y0 + (row_h - h) / 2, CBAR_W, h])
    cb  = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax)
    if row_type == "Mean":
        cb.set_ticks([norm.vmin, (norm.vmin + norm.vmax) / 2, norm.vmax])
        cb.set_ticklabels(["0.0", "0.5", "1.0"])
    else:
        cb.set_ticks([norm.vmin, 0.0, norm.vmax])
        cb.set_ticklabels([f"{norm.vmin:.2f}", "0.00", f"{norm.vmax:.2f}"])
    cb.ax.tick_params(labelsize=CBAR_TICK_SIZE, pad=2, width=0.5, length=3)
    for t in cb.ax.get_yticklabels():
        t.set_fontweight("bold")

# ====================== 预载地图资源 ======================
basins_all = []
if os.path.exists(BASIN_DIR):
    for f in os.listdir(BASIN_DIR):
        if f.endswith(".shp"):
            gdf         = gpd.read_file(os.path.join(BASIN_DIR, f)).to_crs("EPSG:4326")
            basin_name  = f.split('.')[0].lower()
            mapped_abbr = BASIN_NAME_MAP.get(basin_name, basin_name.upper()[:2])
            gdf["abbr"] = mapped_abbr
            basins_all.append(gdf)

basins_gdf  = pd.concat(basins_all, ignore_index=True) if basins_all else gpd.GeoDataFrame()
contour_gdf = None
if os.path.exists(CONTOUR_PATH):
    contour_gdf = gpd.read_file(CONTOUR_PATH).to_crs("EPSG:4326")

DATA_FILES = [
    ("Greenland_albedo_MeanTrend_sig95.csv",          "1950-2024"),
    ("Greenland_1990albedo_MeanTrend_sig95.csv",       "1990-2024"),
    ("Greenland_2001_2024_albedo_MeanTrend_sig95.csv", "2001-2024"),
]
cache = {period: load_and_grid(f) for f, period in DATA_FILES}

all_means  = np.concatenate([v[2].ravel() for v in cache.values()])
mean_norm  = Normalize(vmin=np.nanmin(all_means), vmax=np.nanmax(all_means))
trend_norm = TwoSlopeNorm(vmin=-0.1, vcenter=0.0, vmax=0.1)

colors_mean = ["#0B6623", "#2E8B57", "#66C2A4", "#BFE6C7", "#EAF7EC", "#FFFFFF"]
cmap_mean   = LinearSegmentedColormap.from_list("gw", colors_mean, N=256)

# ====================== 预载时序数据 ======================
upper_order   = ['N', 'NW', 'W', 'SW', 'S', 'SE', 'E', 'NE', 'Entire']
upper_display = ['N', 'NW', 'W', 'SW', 'S', 'SE', 'E', 'NE', 'All']

ts_colors = {
    'E':'#6baed6', 'NE':'#74c476', 'SE':'#9e9ac8', 'Melt':'#fb6a4a', 'N':'#63cdd7',
    'S':'#c9bc36', 'W':'#fdae6b',  'NW':'#a6611a',  'SW':'#e7969c',  'Entire':'#000000'
}

ts_df = pd.DataFrame()
if os.path.exists(TS_CSV_PATH):
    ts_df = pd.read_csv(TS_CSV_PATH)
    ts_df['Variable'] = ts_df['Variable'].replace({'alb': 'albedo'})
    ts_df['Region']   = ts_df['Region'].replace({
        'East':'E', 'EastNorth':'NE', 'EastSouth':'SE', 'Melt':'Melt',
        'North':'N', 'South':'S', 'West':'W', 'WestNorth':'NW', 'WestSouth':'SW',
        'The Entire Greenland Area':'Entire'
    })

ts_sub = ts_df[ts_df['Variable'] == 'albedo'].copy() if not ts_df.empty else pd.DataFrame()

# ====================== 主图绘制 ======================
fig = plt.figure(figsize=(FIG_WIDTH_INCHES, FIG_HEIGHT_INCHES))

# ---------- 行 a/b：地图 ----------
for i, row_type in enumerate(["Mean", "Trend"]):
    current_norm = mean_norm if row_type == "Mean" else trend_norm
    current_cmap = cmap_mean if row_type == "Mean" else "RdBu_r"
    y0 = ROW_Y[i]

    for j, (csv_name, period) in enumerate(DATA_FILES):
        ax = fig.add_axes([COL_X_POS[j], y0, SUBPLOT_W, ROW_H], projection=PROJ)
        ax.set_extent(EXTENT, crs=PC)
        ax.axis("off")

        lats, lons, M, T, S = cache[period]

        if row_type == "Mean":
            ax.pcolormesh(lons, lats, M,
                          cmap=current_cmap, norm=current_norm,
                          transform=PC, zorder=3, shading='auto', rasterized=True)
            if j == 0:
                draw_basins(ax, basins_gdf)
            elif j == 1 and contour_gdf is not None:
                ax.add_geometries(contour_gdf.geometry, crs=PC,
                                  color="darkgrey", linestyle="--", linewidth=0.7,
                                  facecolor='none', zorder=6)
        else:
            T_s = fast_smooth(T)
            lf  = np.linspace(lons.min(), lons.max(), T_s.shape[1])
            tf  = np.linspace(lats.min(), lats.max(), T_s.shape[0])
            ax.pcolormesh(lf, tf, T_s,
                          cmap=current_cmap, norm=current_norm,
                          transform=PC, zorder=3, shading='gouraud', rasterized=True)
            step = DOT_STEP_LASTCOL if j == 2 else DOT_STEP
            mask = (S >= 0.5) & np.isfinite(S)
            yy, xx = np.where(mask[::step, ::step])
            ax.scatter(lons[::step][xx], lats[::step][yy],
                       s=DOT_SIZE, c="black", alpha=DOT_ALPHA,
                       marker=DOT_MARKER, transform=PC, zorder=8, rasterized=True)

        # ---- 列标题只在Mean行（i==0）写，y坐标用ROW_Y[0]+ROW_H，贴近地图顶部 ----
        if i == 0:
            col_cx = COL_X_POS[j] + SUBPLOT_W / 2 - 0.03
            # 【修改】：去掉 + 0.004，改为 - 0.025，让标题下移贴近地图内容
            fig.text(col_cx, ROW_Y[0] + ROW_H - 0.025, period,
                     ha='center', va='bottom',
                     fontsize=FONT_SIZE, fontweight="bold")

    fig.text(ROW_LABEL_X, y0 + ROW_H / 2, row_type,
             fontsize=FONT_SIZE, fontweight="bold", rotation=90, va="center")

    # 【修改】panel letter a/b 分开处理：
    # "a" → Mean行顶部，"b" → Trend行顶部（ROW_Y[1] + ROW_H）
    if i == 0:
        # panel "a"：Mean行顶部
        fig.text(PANEL_LETTER_X, ROW_Y[0] + ROW_H - 0.02,
                 'a', fontsize=PANEL_LETTER_SIZE, fontweight="bold")
    else:
        # panel "b"：Trend行顶部，与"a"写法对称
        fig.text(PANEL_LETTER_X, ROW_Y[1] + ROW_H - 0.04,
                 'b', fontsize=PANEL_LETTER_SIZE, fontweight="bold")

    draw_row_colorbar(fig, y0, ROW_H, current_cmap, current_norm, row_type)

# ---------- 行 c：时序图 ----------
ax_ts_upper = fig.add_axes([TS_LEFT, ts_y_upper, TS_WIDTH, ts_h_upper])
ax_ts_lower = fig.add_axes([TS_LEFT, ts_y_lower, TS_WIDTH, ts_h_lower])

if SHOW_SHADE:
    for ax_shade in [ax_ts_upper, ax_ts_lower]:
        ax_shade.axvspan(2001, 2024,
                         facecolor=SHADE_COLOR, alpha=SHADE_ALPHA,
                         edgecolor='none', zorder=0)

# -- 上子图：9个区域 --
handles_upper = []
if not ts_sub.empty:
    for r in upper_order:
        dat = ts_sub[ts_sub['Region'] == r].sort_values('Year')
        if dat.empty:
            continue
        y_smooth = dat['Value'].rolling(5, center=True, min_periods=1).mean()
        line, = ax_ts_upper.plot(dat['Year'], y_smooth, color=ts_colors[r], lw=1.5)
        ax_ts_upper.fill_between(dat['Year'],
                                 y_smooth - dat['Value'].std(), #/ 4,
                                 y_smooth + dat['Value'].std(), #/ 4,
                                 color=ts_colors[r], alpha=0.1, edgecolor='none')
        handles_upper.append(line)

ax_ts_upper.legend(
    handles_upper, upper_display,
    loc='lower left',
    bbox_to_anchor=(0, 1.02, 1, 0.1),
    mode="expand", ncol=9, frameon=False,
    handlelength=1.2, handletextpad=0.2, borderaxespad=0,
    prop={'weight': 'bold', 'size': TS_FONT}
)

ax_ts_upper.set_yticks([0.65, 0.70, 0.75, 0.80, 0.85])
ax_ts_upper.set_yticklabels(['0.65', '0.70', '0.75', '0.80', '0.85'],
                             fontsize=TS_FONT, fontweight='bold')
ax_ts_upper.set_ylim(0.65, 0.85)
ax_ts_upper.set_xlim(1950, 2024)
ax_ts_upper.margins(x=0)
ax_ts_upper.tick_params(labelbottom=False, direction='in', width=0.8, labelsize=TS_FONT)
ax_ts_upper.set_ylabel('')
for sp in ax_ts_upper.spines.values():
    sp.set_linewidth(0.8)

# -- 下子图：Melt --
if not ts_sub.empty:
    dat_melt = ts_sub[ts_sub['Region'] == 'Melt'].sort_values('Year')
    if not dat_melt.empty:
        y_smooth = dat_melt['Value'].rolling(5, center=True, min_periods=1).mean()
        ax_ts_lower.plot(dat_melt['Year'], y_smooth,
                         color=ts_colors['Melt'], lw=1.5)
        ax_ts_lower.fill_between(dat_melt['Year'],
                                 y_smooth - dat_melt['Value'].std(), #/ 4,
                                 y_smooth + dat_melt['Value'].std(), #/ 4,
                                 color=ts_colors['Melt'], alpha=0.1, edgecolor='none')

ax_ts_lower.set_yticks([0.45, 0.50, 0.55])
ax_ts_lower.set_yticklabels(['0.45', '0.50', '0.55'],
                              fontsize=TS_FONT, fontweight='bold')
ax_ts_lower.set_ylim(0.43, 0.58)
ax_ts_lower.tick_params(direction='out', width=0.8, labelsize=TS_FONT)
ax_ts_lower.set_ylabel('')
for sp in ax_ts_lower.spines.values():
    sp.set_linewidth(0.8)

years_ticks = np.arange(1950, 2024, 15).tolist()
if 2024 not in years_ticks:
    years_ticks.append(2024)
ax_ts_lower.set_xticks(years_ticks)
ax_ts_lower.set_xticklabels([str(y) for y in years_ticks],
                              fontsize=TS_FONT, fontweight='bold')
ax_ts_lower.set_xlim(1950, 2024)
ax_ts_lower.margins(x=0)
ax_ts_lower.set_xlabel('Year', fontsize=TS_FONT, fontweight='bold', labelpad=6)

# Melt 图例
melt_y  = ts_y_upper - TS_INNER_GAP / 2
melt_x0 = TS_LEFT
melt_x1 = TS_LEFT + 0.03
melt_xt = TS_LEFT + 0.035

fig.add_artist(Line2D(
    [melt_x0, melt_x1], [melt_y, melt_y],
    transform=fig.transFigure,
    color=ts_colors['Melt'], linewidth=1.5, solid_capstyle='round'
))
fig.text(melt_xt, melt_y, 'Melt',
         va='center', fontsize=TS_FONT, fontweight='bold', color='black')

# Albedo 行标签
fig.text(ROW_LABEL_X, TS_Y0 + TS_H / 2, 'Albedo',
         va='center', rotation='vertical',
         fontsize=FONT_SIZE, fontweight='bold')

# 面板字母 c
fig.text(PANEL_LETTER_X, TS_Y0 + TS_H + 0.008, 'c',
         fontsize=PANEL_LETTER_SIZE, fontweight='bold')

# ====================== 导出（暂时注释） ======================
out_base = os.path.join(OUT_DIR, "Fig1")
os.makedirs(OUT_DIR, exist_ok=True)
plt.savefig(out_base + ".png", dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.savefig(out_base + ".pdf", dpi=300, bbox_inches="tight", pad_inches=0.02, format="pdf")
plt.savefig(out_base + ".eps", dpi=300, bbox_inches="tight", pad_inches=0.02, format="eps")
print(f"✅ 已导出：\n  {out_base}.png\n  {out_base}.pdf\n  {out_base}.eps")

图2

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

# =========================
# 1. 全局出版级格式设置
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.6,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
})

# =========================
# 2. 配置与数据
# =========================
VAR_DISPLAY_MAP = {
    "T2M": "2m Air Temp.", "STRD": "Downward LW", "SSRD": "Downward SW",
    "TCC": "Cloud Cover", "U10/V10\n(MAX)": "10m Wind Speed"
}

BASIN_ORDER = ["N", "NW", "W", "SW", "S", "SE", "E", "NE", "Melt"]
ALTITUDE_LABELS = ["<500", "500-1000", "1000-1500", "1500-2000", "2000-2500", "2500-3000", ">3000"]
COLORS_BOTTOM = {
    "T2M": "#E63946", "STRD": "#F77F00", "SSRD": "#FCBF49",
    "TCC": "#06A77D", "U10/V10\n(MAX)": "#1D3557"
}
BOTTOM_VARS = ["T2M", "STRD", "SSRD", "TCC", "U10/V10\n(MAX)"]

# 数据准备
df_elev = pd.DataFrame({
    "T2M": [0.5067, 0.5605, 0.3679, 0.0876, 0.0327, 0.0012, 0.0306],
    "STRD": [0.3416, 0.3501, 0.2391, 0.1697, 0.0335, 0.0010, 0.0673],
    "SSRD": [0.2194, 0.1188, 0.0579, 0.0116, 0.0067, 0.0294, 0.0378],
    "TCC": [0.0398, 0.0396, 0.0444, 0.0001, 0.0163, 0.0145, 0.0068],
    "U10": [0.0124, 0.0197, 0.0070, 0.0078, 0.1734, 0.0288, 0.0414],
    "V10": [0.0051, 0.0007, 0.0232, 0.0281, 0.0262, 0.0571, 0.0241]
}, index=ALTITUDE_LABELS)
df_elev["U10/V10\n(MAX)"] = np.maximum(df_elev["U10"], df_elev["V10"])

basin_data = {
    "N": {"T2M": 0.2411, "STRD": 0.4201, "SSRD": 0.0321, "TCC": 0.0691, "U10": 0.0298, "V10": 0.0600},
    "NW": {"T2M": 0.4366, "STRD": 0.3528, "SSRD": 0.0170, "TCC": 0.1051, "U10": 0.1378, "V10": 0.2189},
    "W": {"T2M": 0.4309, "STRD": 0.3130, "SSRD": 0.0119, "TCC": 0.1174, "U10": 0.0521, "V10": 0.0033},
    "SW": {"T2M": 0.6971, "STRD": 0.4691, "SSRD": 0.0725, "TCC": 0.1046, "U10": 0.0698, "V10": 0.0119},
    "S": {"T2M": 0.7280, "STRD": 0.5224, "SSRD": 0.1876, "TCC": 0.0019, "U10": 0.0857, "V10": 0.0031},
    "SE": {"T2M": 0.7040, "STRD": 0.4947, "SSRD": 0.0664, "TCC": 0.0297, "U10": 0.1265, "V10": 0.0097},
    "E": {"T2M": 0.5131, "STRD": 0.5232, "SSRD": 0.0346, "TCC": 0.0307, "U10": 0.0147, "V10": 0.1964},
    "NE": {"T2M": 0.3907, "STRD": 0.4117, "SSRD": 0.0120, "TCC": 0.1441, "U10": 0.0067, "V10": 0.1178},
    "Melt": {"T2M": 0.5384, "STRD": 0.3817, "SSRD": 0.1809, "TCC": 0.0384, "U10": 0.0436, "V10": 0.0072}
}
df_basin = pd.DataFrame(basin_data).T.loc[BASIN_ORDER]
df_basin["U10/V10\n(MAX)"] = np.maximum(df_basin["U10"], df_basin["V10"])

# =========================
# 3. 绘图主程序
# =========================
def draw_transposed_fig():
    fig = plt.figure(figsize=(200/25.4, 3.84), dpi=300)

    # 上方留出图例空间，bottom 收回
    gs = gridspec.GridSpec(1, 2, wspace=0.18, bottom=0.18, top=0.82, left=0.08, right=0.98)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])

    # --- (a) Basin Vertical Bar Chart ---
    x_basin = np.arange(len(df_basin.index))
    bar_w = 0.15
    offsets = np.linspace(-bar_w*2, bar_w*2, 5)

    for i, v in enumerate(BOTTOM_VARS):
        ax_a.bar(x_basin + offsets[i], df_basin[v], width=bar_w,
                 label=VAR_DISPLAY_MAP.get(v, v), color=COLORS_BOTTOM[v],
                 edgecolor="black", linewidth=0.4)

    ax_a.set_xticks(x_basin)
    ax_a.set_xticklabels(df_basin.index, fontweight="bold")
    ax_a.set_xlabel("Basin", fontweight="bold", labelpad=16)
    ax_a.set_ylabel("$R^2$", fontweight="bold")
    ax_a.set_ylim(0, 1.0)
    ax_a.set_yticks([0, 0.5, 1.0])

    # --- (b) Elevation Line Chart ---
    x_elev = np.arange(len(df_elev.index))
    for v in BOTTOM_VARS:
        ax_b.plot(x_elev, df_elev[v].values, marker="o", lw=1.0, ls="--", ms=3.5,
                  label=VAR_DISPLAY_MAP.get(v, v), color=COLORS_BOTTOM[v],
                  markeredgecolor="black", markeredgewidth=0.4)

    ax_b.set_xticks(x_elev)
    ax_b.set_xticklabels(df_elev.index, fontweight="bold", rotation=25, ha="right")
    ax_b.set_xlabel("Elevation (m)", fontweight="bold")
    ax_b.set_ylabel("$R^2$", fontweight="bold")
    ax_b.set_ylim(0, 1.0)
    ax_b.set_yticks([0, 0.5, 1.0])

    # 统一修饰
    for ax, lbl in zip([ax_a, ax_b], ["a", "b"]):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.text(-0.14, 1.02, lbl, transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom")

    # 用 mpatches.Patch 构造实心色块 handle，handlelength/handleheight 控制宽高比约 5:1
    patch_handles = [
        mpatches.Patch(facecolor=COLORS_BOTTOM[v], edgecolor="black", linewidth=0.4,
                       label=VAR_DISPLAY_MAP.get(v, v))
        for v in BOTTOM_VARS
    ]
    fig.legend(handles=patch_handles,
               loc='upper center', bbox_to_anchor=(0.5, 0.98),
               ncol=5, frameon=False,
               prop={"weight": "bold", "size": 9},
               columnspacing=1.2, handletextpad=0.4,
               handlelength=2.5, handleheight=0.05)

    plt.show()

if __name__ == "__main__":
    draw_transposed_fig()

图3

In [ ]:
# -*- coding: utf-8 -*-
import sys
sys.setrecursionlimit(100000)

import os
import string
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import cartopy.crs as ccrs
from cartopy.feature import ShapelyFeature
import geopandas as gpd
import warnings

matplotlib.use("TkAgg")
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family":      "Arial",
    "mathtext.fontset": "cm",
    "font.size":        9,
    "axes.labelsize":   9,
    "xtick.labelsize":  8,
    "ytick.labelsize":  8,
    "legend.fontsize":  8,
    "axes.linewidth":   0.6,
    "font.weight":      "bold",
    "axes.titleweight": "bold",
    "pdf.fonttype":     42,
    "ps.fonttype":      42,
})

DIR_MAIN   = r"C:\greenland\1995bu\fig4"
BASIN_DIR  = r"C:\greenland\Greenland_shp\流域冰盖"
OUTPUT_DIR = r"C:\Users\11823\era5land"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAVE_PATH  = os.path.join(OUTPUT_DIR, "Fig4_Trend_npj_Aligned.pdf")

PROJ     = ccrs.NorthPolarStereo(central_longitude=-45)
DATA_CRS = ccrs.PlateCarree()

# ── 地图投影坐标范围（单位 m） ────────────────────────────────────────
# 这是格陵兰内容的紧凑边界，不做人为扩展
EXTENT = [-833000, 1003000, -3400000, -600000]   # [x0, x1, y0, y1]
extent_w = EXTENT[1] - EXTENT[0]   # 1836000 m
extent_h = EXTENT[3] - EXTENT[2]   # 2800000 m

# 压扁系数：axes 的物理高度缩短为"等比例高度"的 (1 - SQUEEZE)
# cartopy 会保持等比例，所以实际地图会在 axes 内等比例缩放
# 要实现视觉压扁，必须用 ax.set_aspect('auto') 覆盖 cartopy 的自动 aspect
# 然后 axes 的物理宽高比决定地图的视觉变形量
SQUEEZE = 0.20   # 高度压缩 20%，格陵兰视觉变扁

# 地图内容的宽高比（投影坐标）
map_ratio_true = extent_w / extent_h          # ~0.6557（宽/高）
# 压扁后 axes 的目标宽高比（物理坐标）
map_ratio_squeezed = map_ratio_true / (1.0 - SQUEEZE)  # 更宽，即格陵兰视觉变扁

PERIODS = [
    ("1950-2024", ""),
    ("1990-2024", "_1990_2024"),
    ("2001-2024", "_2001_2024"),
]
VARS = ["T2M", "STRD", "SSRD", "TCC", "UV10MAX"]

VAR_TITLES = {
    "T2M":     "2m Air Temp.",
    "STRD":    "Downward LW",
    "SSRD":    "Downward SW",
    "TCC":     "Cloud Cover",
    "UV10MAX": "10m Wind Speed",
}

UNITS = {
    "T2M":     r"K",
    "STRD":    r"$10^{5}$ J m$^{-2}$",
    "SSRD":    r"$10^{5}$ J m$^{-2}$",
    "TCC":     r"%",
    "UV10MAX": r"m s$^{-1}$",
}

PATH_UV10MAX_ORIG = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\Greenland_UV10MAX_MeanTrend.csv"

def get_csv_path(var, suffix):
    if var == "UV10MAX":
        return PATH_UV10MAX_ORIG if suffix == "" else \
               os.path.join(DIR_MAIN, f"Greenland_ws10_MeanTrend{suffix}.csv")
    return os.path.join(DIR_MAIN, f"Greenland_{var.lower()}_MeanTrend{suffix}.csv")

BASIN_NAMES = ["East", "EastNorth", "EastSouth", "North",
               "South", "West", "WestNorth", "WestSouth"]

basin_features = []
for name in BASIN_NAMES:
    shp_path = os.path.join(BASIN_DIR, f"{name}.shp")
    if os.path.exists(shp_path):
        gdf = gpd.read_file(shp_path).to_crs("EPSG:4326")
        gdf.geometry = gdf.geometry.buffer(0)
        basin_features.append(
            ShapelyFeature(
                gdf.geometry,
                DATA_CRS,
                edgecolor="darkgrey",
                facecolor="none",
                linestyle="--",
                linewidth=0.4,
            )
        )

val_col = "trend_per_decade"
grids = {}
for period_label, suffix in PERIODS:
    for var in VARS:
        path = get_csv_path(var, suffix)
        if not os.path.exists(path):
            grids[(period_label, var)] = None
            continue
        df = pd.read_csv(path)
        if var == "TCC":
            df[val_col] *= 100
        if var in ("STRD", "SSRD"):
            df[val_col] /= 1e5
        grids[(period_label, var)] = df.pivot(index="lat", columns="lon", values=val_col)

_raw_lims = {
    "T2M":     (-0.2, 1.6),
    "TCC":     (-4.0, 4.0),
    "STRD":    (-3.0, 4.0),
    "SSRD":    (-3.0, 4.0),
    "UV10MAX": (-1.1, 1.1),
}
norms = {}
for var, (vmin, vmax) in _raw_lims.items():
    sym = max(abs(vmin), abs(vmax))
    norms[var] = TwoSlopeNorm(vmin=-sym, vcenter=0, vmax=sym)

# ── 布局计算 ──────────────────────────────────────────────────────────
N_ROWS, N_COLS = len(VARS), len(PERIODS)   # 5 行 3 列

# 水平方向（figure 归一化坐标）
LEFT      = 0.12   # 左侧行标签空间
CB_ZONE   = 0.07   # 右侧色标区域
WSPACE    = 0.001  # 列间距（极小，仅防止 axes 边框重叠）
RIGHT_EDGE = 1.0 - CB_ZONE

slot_w = (RIGHT_EDGE - LEFT - (N_COLS - 1) * WSPACE) / N_COLS  # 每列宽度

# 垂直方向：
# slot_h 由 slot_w 和压扁后的宽高比决定，使 axes 物理宽/高 = map_ratio_squeezed
# slot_h（figure坐标）= slot_w * FIG_W / (map_ratio_squeezed * FIG_H)
# 先确定 FIG_W，再解 FIG_H，或者反过来
# 策略：先定 FIG_W，由 slot_w 算出 slot_h，再由 N_ROWS*slot_h 算出 FIG_H

FIG_W = 4.2   # 英寸，可调

# slot_w（figure归一化）对应的物理宽度（英寸）
slot_w_inch = slot_w * FIG_W
# 目标 axes 物理高度（英寸），使 axes 宽高比 = map_ratio_squeezed
slot_h_inch = slot_w_inch / map_ratio_squeezed

# 顶部空间（行标题）和底部空间（英寸）
TOP_MARGIN_INCH  = 0.45   # 第一行标题上方
BOT_MARGIN_INCH  = 0.10

FIG_H = TOP_MARGIN_INCH + N_ROWS * slot_h_inch + BOT_MARGIN_INCH

# 转为 figure 归一化坐标
slot_h     = slot_h_inch / FIG_H
TOP_MARGIN = TOP_MARGIN_INCH / FIG_H
BOT_MARGIN = BOT_MARGIN_INCH / FIG_H

# 行起始 y（figure坐标，从上往下）
def row_y(i):
    """第 i 行 axes 的底部 y 坐标（figure归一化，0=底）"""
    return 1.0 - TOP_MARGIN - (i + 1) * slot_h   # 行间距 = 0

# ── 绘图 ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(FIG_W, FIG_H))
im_refs = {}

for i, var in enumerate(VARS):
    y = row_y(i)

    for j, (period_label, suffix) in enumerate(PERIODS):
        x = LEFT + j * (slot_w + WSPACE)

        ax = fig.add_axes([x, y, slot_w, slot_h], projection=PROJ)

        # ★ 关键：覆盖 cartopy 的等比例 aspect，允许 axes 变形
        # 这样 axes 的物理宽高比直接决定地图视觉比例 → 实现压扁
        ax.set_aspect('auto')

        # 设置地图显示范围（投影坐标）
        ax.set_extent(EXTENT, crs=PROJ)

        ax.spines[:].set_visible(False)

        # 第一行：列标题
        if i == 0:
            label = string.ascii_lowercase[j]
            title_x = 0.0 if j == 0 else 0.11   # b/c 列右移
            ax.text(title_x, 1.08, f"{label}  {period_label}",
                    transform=ax.transAxes,
                    fontsize=10,
                    fontweight="bold",
                    va="baseline",
                    ha="left")

        # 第一列：行标签（旋转90°，置于 axes 左侧）
        if j == 0:
            ax.text(-0.18, 0.5, VAR_TITLES[var],
                    transform=ax.transAxes,
                    fontsize=9,
                    fontweight="bold",
                    rotation=90,
                    ha="center",
                    va="center")

        g = grids[(period_label, var)]
        if g is not None:
            im = ax.pcolormesh(
                g.columns,
                g.index,
                g.values,
                transform=DATA_CRS,
                cmap="RdBu_r",
                norm=norms[var],
                shading="auto",
                rasterized=True,
                zorder=3,
            )
            im_refs[var] = im
            for f in basin_features:
                ax.add_feature(f, zorder=5)

# ── 色标 ──────────────────────────────────────────────────────────────
CB_WIDTH  = 0.012   # figure 归一化宽度
CB_LEN_R  = 0.55    # 色标高度占 slot_h 的比例
CB_X      = RIGHT_EDGE + 0.013

for i, var in enumerate(VARS):
    y = row_y(i)
    cb_h  = slot_h * CB_LEN_R
    cb_y  = y + (slot_h - cb_h) / 2

    cb_ax = fig.add_axes([CB_X, cb_y, CB_WIDTH, cb_h])

    sym   = norms[var].vmax
    ticks = [-sym, 0.0, sym]

    if var in im_refs:
        cb = fig.colorbar(im_refs[var],
                          cax=cb_ax,
                          orientation="vertical",
                          ticks=ticks)
        cb.ax.tick_params(labelsize=7.5, length=2, pad=1.5)
        fmt = "{:.0f}" if var == "TCC" else "{:.1f}"
        cb.ax.set_yticklabels(
            [fmt.format(t) if abs(t) > 1e-5 else "0" for t in ticks],
            fontweight="bold"
        )
        cb_ax.set_title(UNITS[var], fontsize=7.5, pad=4, fontweight="bold")

plt.savefig(SAVE_PATH,                       dpi=600, bbox_inches="tight")
plt.savefig(SAVE_PATH.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")

fig.canvas.draw()
plt.show()

print(f"Saved: {SAVE_PATH}")

补充图4


In [ ]:
# -*- coding: utf-8 -*-
"""
Combined publication figure
  (a) SHAP beeswarm  |  (b) Spearman correlation heatmap
npj style, Arial

Layout strategy
───────────────
• Fix SHAP axes height in inches first (n_features rows × row_h).
• Set heatmap CELL_IN so that N_CELLS × CELL_IN == SHAP axes height exactly.
• Place both axes at the same bottom coordinate → tops perfectly aligned.
• Panel labels a/b placed at identical figure-y via fig.text(), not transAxes.
"""

import sys, shap, matplotlib, matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import pandas as pd, joblib, numpy as np
from scipy.stats import rankdata

print(f"Python {sys.version.split()[0]}  |  shap {shap.__version__}  |  mpl {matplotlib.__version__}")

# ── 1. Style ──────────────────────────────────────────────────────────────────
LBL_FS = 11
TK_FS  = 10
PNL_FS = 14
AX_LW  = 0.6

plt.rcParams.update({
    "font.family":        "Arial",
    "font.size":          TK_FS,
    "font.weight":        "bold",
    "axes.labelsize":     LBL_FS,
    "axes.labelweight":   "bold",
    "axes.linewidth":     AX_LW,
    "axes.edgecolor":     "#000000",
    "xtick.direction":    "in",
    "ytick.direction":    "in",
    "xtick.major.width":  AX_LW,
    "ytick.major.width":  AX_LW,
    "xtick.major.size":   2.5,
    "ytick.major.size":   2.5,
    "xtick.labelsize":    TK_FS,
    "ytick.labelsize":    TK_FS,
})

# ── 2. Load data ──────────────────────────────────────────────────────────────
file_path = r"C:\Users\greenlandProject\ERA5LAND\merged_allfeature.csv"
df = pd.read_csv(file_path, usecols=[
    "longitude", "latitude", "altitude_last_valid",
    "sp", "ssrd", "str", "strd",
    "t2m", "u10", "v10", "snowc", "sde", "asn", "year", "month"
])
rename_map = {
    "longitude": "Lon", "latitude": "Lat", "altitude_last_valid": "Alt",
    "sp": "Sp", "ssrd": "Ssrd", "str": "Str", "strd": "Strd",
    "t2m": "T2m", "u10": "U10", "v10": "V10",
    "snowc": "Snowc", "sde": "Sde", "asn": "Asn",
    "year": "Year", "month": "Month",
}
num_df = df.astype(float)
num_df.rename(columns=rename_map, inplace=True)

# ── 3. SHAP ───────────────────────────────────────────────────────────────────
sampled   = num_df.sample(1000, random_state=42)
rf_model  = joblib.load(r"C:\Users\11823\era5land\saved_models_917\RF_917.pkl")
explainer = shap.TreeExplainer(rf_model,
                                feature_perturbation="tree_path_dependent",
                                model_output="raw")
shap_values = explainer.shap_values(sampled, check_additivity=False)

feature_names = list(sampled.columns)
n_features    = len(feature_names)          # 15
order         = np.argsort(np.abs(shap_values).mean(axis=0))

# ── 4. Spearman (fast) ────────────────────────────────────────────────────────
arr   = num_df.values.astype(np.float32)
ranks = np.apply_along_axis(rankdata, 0, arr)
n     = ranks.shape[0]
ranks -= ranks.mean(axis=0)
cov   = (ranks.T @ ranks) / (n - 1)
std   = np.sqrt(np.diag(cov))
corr_arr = cov / np.outer(std, std)
np.clip(corr_arr, -1, 1, out=corr_arr)
corr = pd.DataFrame(corr_arr, index=num_df.columns, columns=num_df.columns)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# ── 5. Exact inch-based layout ────────────────────────────────────────────────
N_CELLS = n_features   # 15

# --- SHAP axes dimensions ---
ROW_H      = 0.25          # inches per feature row  → total height
SHAP_AX_H  = N_CELLS * ROW_H          # = 3.75 in  (same as heatmap)
SHAP_AX_W  = SHAP_AX_H * 0.72         # narrower panel

# --- Heatmap cell size derived so axes height == SHAP_AX_H ---
CELL_IN    = SHAP_AX_H / N_CELLS      # = ROW_H = 0.25 in  → square cells
HMAP_AX    = CELL_IN * N_CELLS        # == SHAP_AX_H  ✓

# --- Padding / spacing (inches) ---
TOP_PAD    = 0.45   # room for panel labels
BOT_PAD    = 0.90   # x-axis labels (rotated 45°)
SHAP_LEFT  = 0.70   # y-tick labels of SHAP
SHAP_RIGHT = 0.10
GAP        = 0.55   # gap between panels
HMAP_LEFT  = 0.60   # y-tick labels of heatmap
CBAR_W     = 0.65   # colorbar + gap
RIGHT_PAD  = 0.15

FIG_W = (SHAP_LEFT + SHAP_AX_W + SHAP_RIGHT +
         GAP +
         HMAP_LEFT + HMAP_AX + CBAR_W + RIGHT_PAD)
FIG_H = TOP_PAD + SHAP_AX_H + BOT_PAD

fig = plt.figure(figsize=(FIG_W, FIG_H))

def to_rect(l, b, w, h):
    """Convert inch coords to figure fraction."""
    return [l/FIG_W, b/FIG_H, w/FIG_W, h/FIG_H]

AXES_BOT = BOT_PAD   # both axes share same bottom → same top

ax_shap = fig.add_axes(to_rect(SHAP_LEFT, AXES_BOT, SHAP_AX_W, SHAP_AX_H))

hmap_l  = SHAP_LEFT + SHAP_AX_W + SHAP_RIGHT + GAP + HMAP_LEFT
ax_corr = fig.add_axes(to_rect(hmap_l, AXES_BOT, HMAP_AX, HMAP_AX))

# Panel labels: placed at same figure-y (top of axes + small offset)
LABEL_Y = (AXES_BOT + SHAP_AX_H + 0.10) / FIG_H   # figure fraction

fig.text(SHAP_LEFT / FIG_W,        LABEL_Y, "a",
         fontsize=PNL_FS, fontweight="bold", va="bottom", ha="left")
fig.text(hmap_l / FIG_W - 0.04,    LABEL_Y, "b",
         fontsize=PNL_FS, fontweight="bold", va="bottom", ha="left")

# ── 6. Panel (a) – manual beeswarm ───────────────────────────────────────────
cmap = cm.get_cmap("coolwarm")
rng  = np.random.default_rng(0)

for plot_pos, feat_idx in enumerate(order):
    sv      = shap_values[:, feat_idx]
    fv      = sampled.iloc[:, feat_idx].values
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-12)
    colors  = cmap(fv_norm)
    jitter  = rng.uniform(-0.28, 0.28, size=len(sv))
    ax_shap.scatter(sv, plot_pos + jitter,
                    c=colors, s=5, alpha=0.6,
                    linewidths=0, rasterized=True)
    ax_shap.axhline(plot_pos, color="gray", lw=0.45, alpha=0.22, zorder=0)

ax_shap.set_yticks(range(n_features))
ax_shap.set_yticklabels([feature_names[i] for i in order], fontsize=TK_FS)
ax_shap.set_ylim(-0.5, n_features - 0.5)
ax_shap.set_xlabel("SHAP value", fontsize=LBL_FS, fontweight="bold")
ax_shap.set_xticks([-0.2, -0.1, 0.0, 0.1])
ax_shap.tick_params(axis="x", labelsize=TK_FS)
ax_shap.axvline(0, color="black", lw=0.5, ls="--", alpha=0.4)

for spine in ax_shap.spines.values():
    spine.set_visible(False)
ax_shap.tick_params(axis="both", length=0)

# ── 7. Panel (b) – heatmap (square=True, axes already square) ────────────────
sns.heatmap(
    corr, mask=mask,
    cmap="coolwarm", vmin=-1, vmax=1, center=0,
    square=True,
    annot=False,
    linewidths=0.25, linecolor="white",
    cbar_kws={"orientation": "vertical",
               "pad": 0.03,
               "shrink": 0.50,
               "aspect": 20,
               "ticks": [-1, -0.5, 0, 0.5, 1]},
    ax=ax_corr,
)
ax_corr.set_xticklabels(ax_corr.get_xticklabels(),
                         rotation=45, ha="right", fontsize=TK_FS)
ax_corr.set_yticklabels(ax_corr.get_yticklabels(),
                         rotation=0,  fontsize=TK_FS)

cbar_b = ax_corr.collections[0].colorbar
cbar_b.set_label("Spearman r", fontsize=LBL_FS, fontweight="bold")
cbar_b.ax.tick_params(labelsize=TK_FS)

# ── 8. Save ───────────────────────────────────────────────────────────────────
out_path = r"C:\Users\11823\era5land\Fig_Combined_SHAP_Corr_npj.png"
plt.savefig(out_path, dpi=400, bbox_inches="tight", transparent=True)
plt.show()
print(f"Saved → {out_path}")

补充图1

In [ ]:
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
from shapely.geometry import mapping
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import os

# ============================================================
# 0. 路径与区域名映射
# ============================================================
nc_path = r"C:\greenland\1995bu\Greenland_albedo1950-2024Summer.nc"
shp_dir = r"C:\greenland\Greenland_shp\流域冰盖"

region_map = {
    'North.shp': 'N',
    'WestNorth.shp': 'NW',
    'West.shp': 'W',
    'WestSouth.shp': 'SW',
    'South.shp': 'S',
    'EastSouth.shp': 'SE',
    'East.shp': 'E',
    'EastNorth.shp': 'NE',
}

# ============================================================
# 1. 读取 nc 数据
# ============================================================
ds = xr.open_dataset(nc_path)
albedo = ds['albedo']
albedo = albedo.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
albedo = albedo.rio.write_crs("EPSG:4326")

years = ds['time'].values
weights_full = np.cos(np.deg2rad(albedo.lat))

# ============================================================
# 2. 区域掩膜 → 面积加权平均时间序列
# ============================================================
region_ts = {}

for shp_file, region_name in region_map.items():
    gdf = gpd.read_file(os.path.join(shp_dir, shp_file))
    clipped = albedo.rio.clip(gdf.geometry.apply(mapping), gdf.crs, drop=True, all_touched=True)
    w = np.cos(np.deg2rad(clipped.lat))
    masked = clipped.where(clipped.notnull())
    weighted_mean = (masked * w).sum(dim=['lat', 'lon']) / (w * masked.notnull()).sum(dim=['lat', 'lon'])
    region_ts[region_name] = weighted_mean.values
    print(f"{region_name}: done, mean = {np.nanmean(weighted_mean.values):.4f}")

gdf_melt = gpd.read_file(os.path.join(shp_dir, 'new', 'melt.shp'))
clipped_melt = albedo.rio.clip(gdf_melt.geometry.apply(mapping), gdf_melt.crs, drop=True, all_touched=True)
w_melt = np.cos(np.deg2rad(clipped_melt.lat))
masked_melt = clipped_melt.where(clipped_melt.notnull())
weighted_mean_melt = (masked_melt * w_melt).sum(dim=['lat', 'lon']) / (w_melt * masked_melt.notnull()).sum(dim=['lat', 'lon'])
region_ts['Melt'] = weighted_mean_melt.values
print(f"Melt: done, mean = {np.nanmean(weighted_mean_melt.values):.4f}")

all_mean = (albedo * weights_full).sum(dim=['lat', 'lon']) / (weights_full * albedo.notnull()).sum(dim=['lat', 'lon'])
region_ts['All'] = all_mean.values
print(f"All: done, mean = {np.nanmean(all_mean.values):.4f}")

# ============================================================
# 3. 线性趋势
# ============================================================
def calc_trend(years, values, start, end):
    mask = (years >= start) & (years <= end) & np.isfinite(values)
    y = years[mask].astype(float)
    v = values[mask]
    if len(v) < 5:
        return np.nan, np.nan, np.nan
    slope, intercept, r, p, se = stats.linregress(y, v)
    ci95 = se * stats.t.ppf(0.975, len(v) - 2)
    return slope * 10, ci95 * 10, p

# ============================================================
# 4. 构建表格数据
# ============================================================
periods = [
    ('1950–2024', 1950, 2024),
    ('1990–2024', 1990, 2024),
    ('2001–2024', 2001, 2024),
]

region_order = ['N', 'NW', 'W', 'SW', 'S', 'SE', 'E', 'NE', 'All', 'Melt']

table_data = []
for reg in region_order:
    ts = np.array(region_ts[reg])
    row = [reg]
    for label, s, e in periods:
        mask = (years >= s) & (years <= e)
        row.append(f"{np.nanmean(ts[mask]):.3f}")
    for label, s, e in periods:
        slope, ci, p = calc_trend(years, ts, s, e)
        sig = '*' if p < 0.05 else ''
        row.append(f"{slope:+.3f}{sig}")
    table_data.append(row)

col_headers = [
    'Region',
    'Mean\n1950–2024', 'Mean\n1990–2024', 'Mean\n2001–2024',
    'Trend\n1950–2024', 'Trend\n1990–2024', 'Trend\n2001–2024',
]

df_table = pd.DataFrame(table_data, columns=[c.replace('\n', ' ') for c in col_headers])
df_table.to_csv(r"C:\greenland\Table_S1_regional_albedo_stats.csv", index=False)
print("\nCSV saved.")

# ============================================================
# 5. 绘图 — 用你的版式
# ============================================================
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.labelsize'] = 10
matplotlib.rcParams['xtick.labelsize'] = 10
matplotlib.rcParams['ytick.labelsize'] = 10
matplotlib.rcParams['legend.fontsize'] = 10

fig = plt.figure(figsize=(6.5, 8), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.2], hspace=0.1)

label_x_f = 0.015
label_ya_f = 0.98
label_yb_f = 0.52

# --- a: Grouped Bar Chart ---
ax_bar = fig.add_subplot(gs[0])
fig.text(label_x_f, label_ya_f, 'a', fontsize=14, fontweight='bold', va='top')

bar_width = 0.22
x = np.arange(len(region_order))
period_colors = ['#4393c3', '#f4a582', '#d6604d']

for i, (label, s, e) in enumerate(periods):
    slopes, cis = [], []
    for reg in region_order:
        ts = np.array(region_ts.get(reg, np.nan))
        slope, ci, p = calc_trend(years, ts, s, e)
        slopes.append(slope if not np.isnan(slope) else 0)
        cis.append(ci if not np.isnan(ci) else 0)
    offset = (i - 1) * bar_width
    ax_bar.bar(x + offset, slopes, bar_width, yerr=cis, capsize=2,
               color=period_colors[i], edgecolor='black', linewidth=0.5,
               label=label, zorder=3)

ax_bar.axhline(y=0, color='black', linewidth=0.8, zorder=2)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(region_order, fontweight='bold')
ax_bar.set_ylabel('Albedo trend')
ax_bar.legend(frameon=False, loc='upper left')
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

# --- b: Table ---
ax_table_area = fig.add_subplot(gs[1])
ax_table_area.axis('off')
fig.text(label_x_f, label_yb_f, 'b', fontsize=14, fontweight='bold', va='top')

x_offset = -0.02
inset_table = inset_axes(ax_table_area, width="100%", height="90%",
                          bbox_to_anchor=(x_offset, 0, 1, 1),
                          bbox_transform=ax_table_area.transAxes,
                          loc='upper left', borderpad=0)
inset_table.axis('off')

table = inset_table.table(
    cellText=table_data,
    colLabels=col_headers,
    cellLoc='center',
    loc='upper left',
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.auto_set_column_width(col=list(range(len(col_headers))))
table.scale(1.1, 2.5)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#cccccc')
    cell.set_linewidth(0.5)
    if row > 0:
        cell.set_height(cell.get_height() * 0.8)
    if row == 0:
        cell.set_facecolor('#e8e8e8')
        cell.get_text().set_weight('bold')
        cell.set_text_props(va='center')
    elif row > 0 and table_data[row - 1][0] in ['All', 'Melt']:
        cell.set_facecolor('#f5f5f5')
    else:
        cell.set_facecolor('white')

# ============================================================
# 6. 保存
# ============================================================
fig.subplots_adjust(left=0.1, right=0.95, top=0.95, bottom=0.05)

fig.savefig(r"C:\greenland\FigS1_regional_stats.png", dpi=300, bbox_inches='tight')
fig.savefig(r"C:\greenland\FigS1_regional_stats.pdf", bbox_inches='tight')
plt.show()
print("Figure saved.")

补充图2

In [ ]:
# -*- coding: utf-8 -*-
import re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# =========================
# 1. 全局出版级格式设置
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "pdf.fonttype": 42,
})

# =========================
# 2. 路径与变量配置
# =========================
TXT_PATH = r"C:\Users\11823\Desktop\regression_output.txt"
OUT_DIR  = r"C:\Users\11823\era5land"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

VAR_DISPLAY_MAP = {
    "T2M": "2m Air Temp.",
    "STRD": "Downward LW",
    "SSRD": "Downward SW",
    "STR": "Net Thermal Rad.",
    "TCC": "Cloud Cover",
    "RF": "Runoff",
    "SF": "Snowfall",
    "U10": "10m U-Wind",
    "V10": "10m V-Wind",
    "U10/V10\n(MAX)": "10m Wind Speed"
}

PRIORITY = [
    "t2m", "strd", "ssrd", "str",
    "tcc", "rf", "sf", "u10", "v10"
]

BASIN_ORDER = ["N", "NW", "W", "SW", "S", "SE", "E", "NE", "Melt"]

# =========================
# 3. 函数定义
# =========================
def parse_univariate_blocks(txt: str):

    block_pat = re.compile(
        r"^\[(?P<kind>Altitude|Basin)\s+(?P<group>[^\]]+)\]\s+---\s+Univariate\s*\n"
        r"^变量\s*\|\s*slope\s*\|\s*R2\s*\|\s*r\s*\|\s*n\s*\n"
        r"(?P<body>(?:.*\n)*?)(?=^\[|\Z|^=+|^-{5,})",
        flags=re.MULTILINE
    )

    row_pat = re.compile(
        r"^\s*(?P<var>\S+)\s*\|\s*"
        r"(?P<slope>[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*\|\s*"
        r"(?P<R2>[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*\|\s*"
        r"(?P<r>[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\s*\|\s*"
        r"(?P<n>\d[\d,]*)\s*$",
        flags=re.MULTILINE
    )

    rows = []

    for bm in block_pat.finditer(txt):

        kind  = bm.group("kind")
        group = bm.group("group")

        for rm in row_pat.finditer(bm.group("body")):

            rows.append({
                "kind": kind,
                "group": group,
                "var": rm.group("var"),
                "R2": float(rm.group("R2"))
            })

    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    return (
        df[df.kind == "Altitude"],
        df[df.kind == "Basin"]
    )

def build_var_order(df):

    if df.empty:
        return PRIORITY

    d = df.copy()
    d["var_l"] = d["var"].str.lower()

    allv = list(
        pd.unique(
            d[~d["var_l"].isin({"sp", "slhf", "tp"})]["var_l"]
        )
    )

    ordered = [v for v in PRIORITY if v in allv]

    return ordered + sorted(v for v in allv if v not in ordered)

def format_altitude_label(g):

    g = g.lower().replace(" ", "").replace("m", "")

    if g.endswith("+"):
        return f">{g.replace('+', '')} m"

    if "-" in g:
        a, b = g.split("-")
        return f"<{b}" if a == "0" else f"{a}-{b}"

    return g

def draw_boxplot_panel_h(ax, df, kind, var_order, legend_label_func=None):

    if df.empty:
        return

    d = df.copy()
    d["var_l"] = d["var"].str.lower()

    d = d[d["var_l"].isin(var_order)]

    groups = (
        [
            "0-500m", "500-1000m", "1000-1500m",
            "1500-2000m", "2000-2500m",
            "2500-3000m", "3000m+"
        ]
        if kind == "Altitude"
        else BASIN_ORDER
    )

    groups = [g for g in groups if g in d.group.unique()]

    cmap = plt.get_cmap("tab20")
    g2c = {g: cmap(i % 20) for i, g in enumerate(groups)}

    display_order = var_order[::-1]

    data = [
        d[d.var_l == v].R2.values
        for v in display_order
    ]

    bp = ax.boxplot(
        data,
        vert=False,
        whis=(5, 95),
        widths=0.45,
        patch_artist=True,
        showfliers=False
    )

    for b in bp["boxes"]:
        b.set(
            facecolor=(0, 0, 0, 0.05),
            edgecolor="black",
            linewidth=0.6
        )

    for m in bp["medians"]:
        m.set(color="black", linewidth=0.8)

    ys = np.arange(1, len(display_order) + 1)

    for g in groups:

        sg = d[d.group == g]

        for i, v in enumerate(display_order):

            vv = sg[sg.var_l == v].R2

            if len(vv):

                ax.scatter(
                    vv.iloc[0],
                    ys[i],
                    s=12,
                    color=g2c[g],
                    edgecolors="black",
                    linewidths=0.4,
                    label=g,
                    zorder=3
                )

    ax.set_yticks(ys)

    ax.set_yticklabels(
        [
            VAR_DISPLAY_MAP.get(v.upper(), v.upper())
            for v in display_order
        ],
        fontweight="bold"
    )

    ax.set_xlabel("R²", fontweight="bold")
    ax.set_xlim(0, 1.0)

    handles, labels = ax.get_legend_handles_labels()

    by_label = dict(zip(labels, handles))

    labels = list(by_label.keys())

    if legend_label_func:
        labels = [legend_label_func(l) for l in labels]

    ax.legend(
        by_label.values(),
        labels,
        frameon=False,
        loc="lower left",
        bbox_to_anchor=(-0.05, 1.02),
        ncol=3,
        handletextpad=0.15,
        columnspacing=0.5,
        prop={"weight": "bold", "size": 9}
    )

# =========================
# 4. 主程序
# =========================
def main():

    alt_df = pd.DataFrame()
    bas_df = pd.DataFrame()

    box_vars = PRIORITY

    if os.path.exists(TXT_PATH):

        with open(TXT_PATH, "r", encoding="utf-8") as f:

            alt_df, bas_df = parse_univariate_blocks(f.read())

        if not alt_df.empty or not bas_df.empty:

            box_vars = build_var_order(
                pd.concat([alt_df, bas_df])
            )

    # =========================
    # Fig 2 (a, b)
    # =========================
    fig2 = plt.figure(
        figsize=(183/25.4, 6.2),
        dpi=300
    )

    gs2 = gridspec.GridSpec(
        1, 2,
        wspace=0.45
    )

    ax_a = fig2.add_subplot(gs2[0, 0])
    ax_b = fig2.add_subplot(gs2[0, 1])

    draw_boxplot_panel_h(
        ax_a,
        bas_df,
        "Basin",
        box_vars
    )

    draw_boxplot_panel_h(
        ax_b,
        alt_df,
        "Altitude",
        box_vars,
        legend_label_func=format_altitude_label
    )

    for ax, lbl in zip([ax_a, ax_b], ["a", "b"]):

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.annotate(
            lbl,
            xy=(-0.15, 1.18),
            xycoords="axes fraction",
            fontsize=14,
            fontweight="bold",
            va="bottom",
            ha="right"
        )

    fig2.tight_layout(rect=[0, 0, 1, 0.95])

    fig2.savefig(
        os.path.join(OUT_DIR, "Fig2_Boxplots.pdf"),
        bbox_inches="tight"
    )

    plt.show()

if __name__ == "__main__":
    main()

补充图3

In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter
import cartopy.crs as ccrs
import geopandas as gpd

# =====================================================================
# 1. npj 官方标准全局配置 (增大字号版本)
# =====================================================================
plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        10,
    "axes.labelsize":   10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.linewidth":   0.6,
    "font.weight":      "bold",
    "axes.titleweight": "bold",
    "pdf.fonttype":     42,
    "ps.fonttype":      42,
})

# =====================================================================
# 路径与投影
# =====================================================================
DATA_DIR  = r"C:\greenland\1995bu\fig4"
BASIN_DIR = r"C:\greenland\Greenland_shp\流域冰盖"

PROJ     = ccrs.NorthPolarStereo(central_longitude=-45)
DATA_CRS = ccrs.PlateCarree()
EXTENT   = [-660000, 830000, -3300000, -680000]

VLIM = 10.0  
CMAP = "RdBu_r"

PERIODS = [
    ("1950-2024", ""),
    ("1990-2024", "_1990_2024"),
    ("2001-2024", "_2001_2024"),
]

COL_CONFIG = [
    {"label": "a", "title": "ERA5 Net-rad Trend", "key": "fal"},
    {"label": "b", "title": "Model Net-rad Trend", "key": "albedo"},
]

# =====================================================================
# 2. 画布与布局
# =====================================================================
fig_w = 4.0  
fig_h = 7.8  
fig = plt.figure(figsize=(fig_w, fig_h))

gs = fig.add_gridspec(
    3, 2, 
    left=0.15, right=0.98, 
    bottom=0.12, top=0.94, 
    wspace=-0.2, hspace=0.04 
)

# 加载流域边界
basins_gdf = None
if os.path.exists(BASIN_DIR):
    shp_files = [f for f in os.listdir(BASIN_DIR) if f.endswith(".shp")]
    if shp_files:
        basins_all = [gpd.read_file(os.path.join(BASIN_DIR, f)).to_crs(DATA_CRS) for f in shp_files]
        basins_gdf = pd.concat(basins_all, ignore_index=True)

# =====================================================================
# 3. 循环绘制
# =====================================================================
for i, (period_label, suffix) in enumerate(PERIODS):
    for j, cfg in enumerate(COL_CONFIG):
        
        method_key = cfg["key"]
        csv_path = os.path.join(DATA_DIR, f"Greenland_netrad_{method_key}_MeanTrend{suffix}.csv")
        
        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)
        ax = fig.add_subplot(gs[i, j], projection=PROJ)
        ax.set_extent(EXTENT, crs=PROJ)
        ax.axis('off')

        # 数据平滑
        df['plot_val'] = df['trend_per_decade'] / 1e5
        grid = df.pivot(index="lat", columns="lon", values="plot_val").sort_index()
        vals = grid.values.astype(float)
        nan_mask = np.isnan(vals)
        vals_smooth = gaussian_filter(np.where(nan_mask, 0, vals), sigma=0.8)
        vals_smooth[nan_mask] = np.nan

        # 绘图
        norm = TwoSlopeNorm(vmin=-VLIM, vcenter=0, vmax=VLIM)
        im = ax.pcolormesh(
            grid.columns, grid.index, vals_smooth,
            transform=DATA_CRS, cmap=CMAP, norm=norm,
            shading="gouraud", rasterized=True
        )

        if basins_gdf is not None:
            ax.add_geometries(
                basins_gdf.geometry, crs=DATA_CRS,
                edgecolor="black", facecolor="none", 
                linewidth=0.3, zorder=5, alpha=0.5
            )

        # -------------------------------------------------------------
        # 4. 标题与编号（左移 + b单独右移）
        # -------------------------------------------------------------
        if i == 0:
            x0 = -0.16
            dx = 0.16
            extra_shift = 0.04 if cfg["label"] == "b" else 0.0

            ax.text(x0 + extra_shift, 1.02, cfg["label"],
                    transform=ax.transAxes,
                    fontsize=12, fontweight='bold',
                    va='bottom', ha='left')

            ax.text(x0 + dx + extra_shift, 1.02, cfg["title"],
                    transform=ax.transAxes,
                    fontsize=9, fontweight='bold',
                    va='bottom', ha='left')

        # -------------------------------------------------------------
        # 5. 行标签
        # -------------------------------------------------------------
        if j == 0:
            ax.text(-0.08, 0.5, period_label, transform=ax.transAxes, 
                    rotation=90, va='center', ha='right',
                    fontsize=10, fontweight='bold')

# =====================================================================
# 6. 色标
# =====================================================================
cb_ax = fig.add_axes([0.3, 0.08, 0.4, 0.015]) 
cb = fig.colorbar(im, cax=cb_ax, orientation='horizontal', ticks=[-VLIM, 0, VLIM])
cb.ax.set_xticklabels([f"-{int(VLIM)}", "0", f"{int(VLIM)}"], fontsize=9, fontweight='bold')
cb.outline.set_linewidth(0.6)
cb.ax.tick_params(size=2) 

fig.text(0.5, 0.04, r"Trend ($\times10^5$ $\bf{J\,m^{-2}}$)", 
         ha='center', fontsize=10, fontweight='bold')

# =====================================================================
# 保存
# =====================================================================
OUTPUT = r"C:\Users\11823\era5land\SIfig5"
plt.savefig(f"{OUTPUT}.png", dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.savefig(f"{OUTPUT}.pdf", dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.savefig(f"{OUTPUT}.eps", dpi=600, bbox_inches="tight", pad_inches=0.01)

print(f"Figures saved successfully to: {os.path.dirname(OUTPUT)}")
plt.show()

补充图5

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === 全局字体与风格 ===
fontsize_label = 18
fontsize_tick = 13
fontsize_legend = 13

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['xtick.major.size'] = 4
plt.rcParams['ytick.major.size'] = 4
plt.rcParams['xtick.labelsize'] = fontsize_tick
plt.rcParams['ytick.labelsize'] = fontsize_tick

# ==== 数据加载 ====
df = pd.read_csv(r'C:\Users\11823\era5land\pic\era5_FAL_jja_timeseries.csv')

# === 颜色配置 ===
colors = {
    'Stacking model': '#005AC8',
    'era5land': '#6A6A6A',
    'RACMO2': '#E69F00',
    'MODIS10A1': '#B00020'
}

# =========================
# 单图：Albedo 时间序列
# =========================
fig, ax = plt.subplots(1, 1, figsize=(8.5, 5), dpi=300)

data_cols = ['Stacking model', 'era5land', 'RACMO2', 'MODIS10A1']
labels = ['Stacking model', 'ERA5-Land', 'RACMO2', 'MODIS10A1']
cols_colors = [colors[k] for k in data_cols]

for col, label, color in zip(data_cols, labels, cols_colors):
    ax.plot(df['Year'], df[col], color=color, lw=1.8, alpha=0.9, label=label)

# ===== 轴控制（关键修改）=====
year_start = 1950
year_end = 2024

ax.set_xlim(year_start, year_end)                 # 1950 在最左，2024 在最右
ax.set_xticks([1950, 1970, 1990, 2010, 2024])      # 明确标出 1950 和 2024

ax.set_xlabel('Year', fontsize=fontsize_label, fontweight='bold')
ax.set_ylabel('Albedo', fontsize=fontsize_label, fontweight='bold')

ax.set_ylim(0.68, 0.85)
ax.set_yticks(np.arange(0.70, 0.86, 0.05))

# ===== Legend =====
ax.legend(
    loc='upper left',
    bbox_to_anchor=(0.01, 0.99),
    ncol=4,
    frameon=True,
    fontsize=fontsize_legend,
    edgecolor='black',
    framealpha=1,
    fancybox=False
)

ax.grid(False)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# ===== 压缩留白 =====
fig.subplots_adjust(left=0.08, right=0.99, top=0.92, bottom=0.22)
out_path = r"C:\Users\11823\era5land\SIFig1_Final"
for ext in ['png', 'pdf', 'eps']:
    fig.savefig(f"{out_path}.{ext}", dpi=300, bbox_inches="tight")
plt.show()
